# income_v12_stronger_local_shift

Один финальный файл: `submission.csv`.

Версия v12 берёт сильную базу v10, но не добавляет ещё пачку шумных моделей. Основной упор — на переносимость train→test:

- больше OOF-месяцев для калибровки (`Mar–Jun` вместо только последних трёх месяцев);
- test-like веса для blend/stacking;
- source × risk × prediction-bin калибровка;
- новая local KNN residual/ratio calibration по похожим OOF-клиентам;
- external-label признаки (`label_Below_50k_share_r1`, `label_500k_to_1M_share_r1`, `label_Above_1M_share_r1`, `incomeValueCategory`) используются напрямую в локальной калибровке;
- на выходе создаётся только `submission.csv`.


**v12:** усиливает локальную калибровку: расширенный поиск силы до 1.0, weighted-quantile KNN residual вместо только среднего residual, recency-веса внутри local calibration. На выходе только `submission.csv`.

In [ ]:
# Если пакеты не установлены, раскомментируй и выполни один раз:
# !pip install -q numpy pandas scipy lightgbm scikit-learn

In [1]:
import json
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor, LGBMClassifier
from scipy.optimize import minimize
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# =====================
# Основные переключатели
# =====================
RANDOM_STATE = 42
FAST_MODE = False       # True = быстрый тест ноутбука, False = финальный прогон
DEV_SAMPLE = False      # True = маленькая выборка для проверки синтаксиса. Для submit обязательно False.

# Валидационные фолды для OOF/stacking: последние 3 месяца train.
N_STACK_VAL_DATES = 4 if not FAST_MODE else 2
N_RECENT_DATES = 3
N_RISK_RECENT_DATES = 4
QUANTILE_GRID = [0.50, 0.525, 0.55, 0.575] if not FAST_MODE else [0.55]

N_ESTIMATORS_MAX = 1500 if FAST_MODE else 5200
EARLY_STOPPING_ROUNDS = 80 if FAST_MODE else 180
MAX_BIN = 127 if FAST_MODE else 255

# Для финального обучения без holdout берём средний best_iter с запасом.
FINAL_ITER_MULT = 1.12

# Source-MoE switch. В v11 не держим все soft-expert predictions в stacking, только switch.
MOE_MIN_TRAIN_ROWS = 700 if FAST_MODE else 1800
MOE_N_ESTIMATORS_MAX = 750 if FAST_MODE else 2400

# Risk-MoE эксперты для WMAE-хвостов.
RISK_N_ESTIMATORS_MAX = 850 if FAST_MODE else 2600
RISK_MIN_TRAIN_ROWS = 2500 if not FAST_MODE else 700

# Насколько усиливать train-строки, похожие на test. 0 = выключить.
DOMAIN_WEIGHT_STRENGTH = 0.55

# Абсолютные tail-пороги. Они специально простые: дают стабильные gating-признаки для WMAE.
TAIL_THRESHOLDS = {
    "low40": 40000.0,
    "low50": 50000.0,
    "high150": 150000.0,
    "high250": 250000.0,
    "high400": 400000.0,
    "high700": 700000.0,
}


# Local KNN calibration: исправляет локальные residual/ratio по похожим OOF-клиентам.
LOCAL_K = 280 if not FAST_MODE else 90
LOCAL_CHUNK_SIZE = 4000
LOCAL_RESID_CLIP = 46000.0
LOCAL_LOG_RESID_CLIP = 0.42
LOCAL_MIN_TRAIN_ROWS = 4500 if not FAST_MODE else 900
LOCAL_RECENCY_STRENGTH = 0.45

# Один итоговый файл. Не плодим submission_v11_*.csv.
SUBMISSION_NAME = "submission.csv"
PARAMS_NAME = "fitted_params_v12_stronger_local_shift.json"
print("FAST_MODE =", FAST_MODE, "| DEV_SAMPLE =", DEV_SAMPLE)


FAST_MODE = False | DEV_SAMPLE = False


In [2]:
def find_data_dir():
    """Ищет train/test/sample_submission. Поддерживает папку, /mnt/data и train.zip."""
    candidates = [Path("."), Path("./train_unzip"), Path("/mnt/data/train_unzip"), Path("/mnt/data")]
    for d in candidates:
        if (d / "train.csv").exists() and (d / "test.csv").exists() and (d / "sample_submission.csv").exists():
            return d

    for d in [Path("."), Path("/mnt/data")]:
        z = d / "train.zip"
        if z.exists():
            out = d / "train_unzip"
            out.mkdir(exist_ok=True)
            print("Распаковываю", z, "->", out)
            with zipfile.ZipFile(z, "r") as zz:
                zz.extractall(out)
            if (out / "train.csv").exists() and (out / "test.csv").exists():
                return out
    raise FileNotFoundError("Не нашла train.csv/test.csv/sample_submission.csv или train.zip")

DATA_DIR = find_data_dir()
PARAMS_PATH = Path(PARAMS_NAME)
print("DATA_DIR =", DATA_DIR.resolve())

DATA_DIR = C:\Users\Tatya\Downloads\Telegram Desktop\Практика 1


In [3]:
train = pd.read_csv(DATA_DIR / "train.csv", sep=";", decimal=",", low_memory=False)
test = pd.read_csv(DATA_DIR / "test.csv", sep=";", decimal=",", low_memory=False)
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", sep=";", decimal=",")

if DEV_SAMPLE:
    # Стратифицированный мини-сэмпл по датам, только для проверки работы ноутбука.
    train = (train.groupby("dt", group_keys=False)
             .apply(lambda x: x.sample(min(len(x), 1500), random_state=RANDOM_STATE))
             .reset_index(drop=True))
    test = test.sample(min(len(test), 3000), random_state=RANDOM_STATE).reset_index(drop=True)
    sample_submission = sample_submission[sample_submission["id"].isin(test["id"])].reset_index(drop=True)

print("train:", train.shape)
print("test :", test.shape)
print("sample_submission:", sample_submission.shape)
display(train[["id", "dt", "target", "w"]].head())
print("train dates:\n", train["dt"].value_counts().sort_index())
print("test dates:\n", test["dt"].value_counts().sort_index())

if "first_salary_income" in train.columns and "first_salary_income" in test.columns:
    print(
        "first_salary_income missing:",
        f"train={train['first_salary_income'].isna().mean():.3f}",
        f"test={test['first_salary_income'].isna().mean():.3f}",
    )

train: (76786, 224)
test : (73214, 222)
sample_submission: (73214, 2)


,id,dt,target,w
0,2,2024-04-30,109324.476325,0.301217
1,4,2024-02-29,25558.028662,0.695800
2,5,2024-02-29,40666.753098,0.515970
3,6,2024-04-30,43856.672058,0.478003
4,7,2024-04-30,130420.851992,0.552314


train dates:
 dt
2024-01-31     7243
2024-02-29     8865
2024-03-31    13413
2024-04-30    14858
2024-05-31    16193
2024-06-30    16214
Name: count, dtype: int64
test dates:
 dt
2024-07-31    16795
2024-08-31    14646
2024-09-30    15501
2024-10-31    12598
2024-11-30    13674
Name: count, dtype: int64
first_salary_income missing: train=0.887 test=1.000


In [4]:
# =====================
# Метрика и утилиты
# =====================

def weighted_mean_absolute_error(y_true, y_pred, weights):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def weighted_median(values, weights):
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]
    if len(values) == 0:
        return np.nan
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cw = np.cumsum(weights)
    idx = min(int(np.searchsorted(cw, cw[-1] / 2.0)), len(values) - 1)
    return float(values[idx])


def weighted_quantile(values, q, weights):
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]
    if len(values) == 0:
        return np.nan
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cw = np.cumsum(weights)
    idx = min(int(np.searchsorted(cw, q * cw[-1])), len(values) - 1)
    return float(values[idx])


def weighted_quantile_many(values, qs, weights):
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]
    if len(values) == 0:
        return np.full(len(qs), np.nan)
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cw = np.cumsum(weights)
    prob = cw / cw[-1]
    return np.interp(qs, prob, values)


def safe_div(a, b):
    out = np.asarray(a, dtype=np.float64) / np.where(np.asarray(b, dtype=np.float64) == 0, np.nan, np.asarray(b, dtype=np.float64))
    out = np.where(np.isfinite(out), out, np.nan)
    return out.astype("float32")


def dt_ordinals(series, fallback_ord):
    d = pd.to_datetime(series, errors="coerce")
    return d.map(lambda t: float(t.toordinal()) if pd.notna(t) else float(fallback_ord)).to_numpy()


def time_trend_factor(train_dt_part, y_part, w_part, target_ords,
                      min_dates=4, clip_lo=0.97, clip_hi=1.12):
    dt = pd.to_datetime(train_dt_part)
    dates = np.sort(dt.unique())
    target_ords = np.asarray(target_ords, dtype=np.float64)
    if len(dates) < min_dates:
        return np.ones_like(target_ords)

    meds, ords = [], []
    for d in dates:
        m = (dt == d).values
        meds.append(max(weighted_median(y_part[m], w_part[m]), 1.0))
        ords.append(float(pd.Timestamp(d).toordinal()))
    slope = np.polyfit(ords, np.log(meds), 1)[0]
    last_ord = ords[-1]
    factor = np.exp(slope * (target_ords - last_ord))
    factor[target_ords <= last_ord] = 1.0
    return np.clip(factor, clip_lo, clip_hi)

In [5]:
# =====================
# Feature engineering: test-safe proxy/source признаки + базовые v3/v6 признаки
# =====================

DATE_COLS = ["dt", "period_last_act_ad"]


def first_existing(cols, frame):
    return [c for c in cols if c in frame.columns]


def positive_series(full, col):
    if col not in full.columns:
        return pd.Series(np.nan, index=full.index, dtype="float64")
    s = pd.to_numeric(full[col], errors="coerce").astype("float64")
    return s.where(s > 0)


def first_positive_by_order(full, cols, divide_by=None):
    """Возвращает первый положительный proxy по приоритету cols + код источника."""
    n = len(full)
    out = np.full(n, np.nan, dtype="float64")
    src = np.full(n, -1.0, dtype="float32")
    cnt = np.zeros(n, dtype="float32")
    for j, c in enumerate(cols):
        if c not in full.columns:
            continue
        s = positive_series(full, c).to_numpy(dtype="float64")
        if divide_by and c in divide_by:
            s = s / float(divide_by[c])
        cnt += np.isfinite(s).astype("float32")
        take = np.isnan(out) & np.isfinite(s) & (s > 0)
        out[take] = s[take]
        src[take] = float(j)
    return out, src, cnt


def add_proxy_features(full):
    """Строит test-safe best_income_proxy и source_segment. first_salary_income не используется как proxy."""
    n = len(full)

    # 1) Train-only first_salary: только отдельные признаки/флаги, не best proxy и не source для test.
    first_salary = positive_series(full, "first_salary_income").to_numpy(dtype="float64") if "first_salary_income" in full.columns else np.full(n, np.nan)
    full["proxy_first_salary_value"] = first_salary.astype("float32")
    full["has_first_salary_trainonly"] = np.isfinite(first_salary).astype("float32")
    full["proxy_first_salary_log1p"] = np.log1p(np.nan_to_num(first_salary, nan=0.0)).astype("float32")

    # 2) Salary proxy: только salary_6to12m_avg, потому что он есть и в train, и в test.
    salary_val, salary_src, salary_cnt = first_positive_by_order(full, ["salary_6to12m_avg"])
    full["proxy_salary_value"] = salary_val.astype("float32")
    full["proxy_salary_count"] = salary_cnt.astype("float32")
    full["proxy_salary_log1p"] = np.log1p(np.nan_to_num(salary_val, nan=0.0)).astype("float32")

    # 3) Payout proxy: max не мешаем в proxy, sum_3_month приводим к месячному масштабу.
    payout_cols = [
        "dp_payoutincomedata_payout_avg_6_month",
        "dp_payoutincomedata_payout_avg_3_month",
        "dp_payoutincomedata_payout_avg_prev_year",
        "dp_payoutincomedata_payout_sum_3_month",
    ]
    payout_div = {"dp_payoutincomedata_payout_sum_3_month": 3.0}
    payout_val, payout_src, payout_cnt = first_positive_by_order(full, payout_cols, divide_by=payout_div)
    full["proxy_payout_value"] = payout_val.astype("float32")
    full["proxy_payout_from"] = payout_src.astype("float32")
    full["proxy_payout_count"] = payout_cnt.astype("float32")
    full["proxy_payout_log1p"] = np.log1p(np.nan_to_num(payout_val, nan=0.0)).astype("float32")

    # Отдельные payout features: max/sum не как proxy, а как context.
    if "dp_payoutincomedata_payout_sum_3_month" in full.columns:
        full["payout_sum3_monthly"] = (positive_series(full, "dp_payoutincomedata_payout_sum_3_month") / 3.0).astype("float32")
    for c in ["dp_payoutincomedata_payout_max_3_month", "dp_payoutincomedata_payout_max_6_month"]:
        if c in full.columns:
            full[c + "_to_payout_proxy"] = safe_div(positive_series(full, c), payout_val)

    # 4) ILS salary и ILS payment разделены: это разные шкалы.
    ils_salary_cols = ["dp_ils_avg_salary_1y", "dp_ils_avg_salary_2y", "dp_ils_avg_salary_3y"]
    ils_salary_val, ils_salary_src, ils_salary_cnt = first_positive_by_order(full, ils_salary_cols)
    full["proxy_ils_salary_value"] = ils_salary_val.astype("float32")
    full["proxy_ils_salary_from"] = ils_salary_src.astype("float32")
    full["proxy_ils_salary_count"] = ils_salary_cnt.astype("float32")
    full["proxy_ils_salary_log1p"] = np.log1p(np.nan_to_num(ils_salary_val, nan=0.0)).astype("float32")

    ils_payment_cols = [
        "dp_ils_paymentssum_month_avg",
        "dp_ils_paymentssum_avg_12m",
        "dp_ils_paymentssum_avg_6m",
        "dp_ils_paymentssum_avg_6m_current",
        "dp_ils_accpayment_month_avg",
        "dp_ils_accpayment_avg_12m",
        "dp_ils_accpayment_avg_6m",
        "dp_ils_accpayment_avg_6m_current",
        "dp_ils_accpayment_avg_3m",
    ]
    ils_payment_val, ils_payment_src, ils_payment_cnt = first_positive_by_order(full, ils_payment_cols)
    full["proxy_ils_payment_value"] = ils_payment_val.astype("float32")
    full["proxy_ils_payment_from"] = ils_payment_src.astype("float32")
    full["proxy_ils_payment_count"] = ils_payment_cnt.astype("float32")
    full["proxy_ils_payment_log1p"] = np.log1p(np.nan_to_num(ils_payment_val, nan=0.0)).astype("float32")

    # 5) incomeValue и geo fallback.
    income_val, income_src, income_cnt = first_positive_by_order(full, ["incomeValue"])
    geo_val, geo_src, geo_cnt = first_positive_by_order(full, ["salary_median_in_gex_r1", "per_capita_income_rur_amt"])
    full["proxy_income_value"] = income_val.astype("float32")
    full["proxy_income_log1p"] = np.log1p(np.nan_to_num(income_val, nan=0.0)).astype("float32")
    full["proxy_geo_income_value"] = geo_val.astype("float32")
    full["proxy_geo_income_from"] = geo_src.astype("float32")
    full["proxy_geo_income_log1p"] = np.log1p(np.nan_to_num(geo_val, nan=0.0)).astype("float32")

    # Best proxy строго test-safe, по убыванию доверия.
    # source_segment: 0=no/weak, 1=geo, 2=incomeValue, 3=ILS payment, 4=ILS salary, 5=payout, 6=salary
    best_proxy = np.full(n, np.nan, dtype="float64")
    proxy_type = np.full(n, -1.0, dtype="float32")
    proxy_conf = np.zeros(n, dtype="float32")
    proxy_count = np.zeros(n, dtype="float32")

    ordered_sources = [
        ("salary", 6, 0.92, salary_val, salary_cnt),
        ("payout", 5, 0.84, payout_val, payout_cnt),
        ("ils_salary", 4, 0.76, ils_salary_val, ils_salary_cnt),
        ("ils_payment", 3, 0.62, ils_payment_val, ils_payment_cnt),
        ("income_value", 2, 0.48, income_val, income_cnt),
        ("geo_income", 1, 0.25, geo_val, geo_cnt),
    ]
    for name, code, conf, val, cnt in ordered_sources:
        take = np.isnan(best_proxy) & np.isfinite(val) & (val > 0)
        best_proxy[take] = val[take]
        proxy_type[take] = float(code)
        proxy_conf[take] = float(conf)
        proxy_count[take] = cnt[take]

    full["best_income_proxy"] = best_proxy.astype("float32")
    full["best_income_proxy_log1p"] = np.log1p(np.nan_to_num(best_proxy, nan=0.0)).astype("float32")
    full["proxy_type_code"] = proxy_type.astype("float32")
    full["proxy_confidence"] = proxy_conf.astype("float32")
    full["proxy_source_count"] = proxy_count.astype("float32")
    full["has_good_proxy"] = ((proxy_type >= 3) & np.isfinite(best_proxy)).astype("float32")
    full["source_segment"] = np.maximum(proxy_type, 0).astype("float32")

    # Согласованность между источниками proxy.
    compare_pairs = [
        ("proxy_salary_value", "proxy_payout_value", "ratio_salary_payout_proxy"),
        ("proxy_salary_value", "proxy_ils_salary_value", "ratio_salary_ils_salary_proxy"),
        ("proxy_payout_value", "proxy_income_value", "ratio_payout_income_proxy"),
        ("proxy_ils_salary_value", "proxy_income_value", "ratio_ils_salary_income_proxy"),
        ("proxy_ils_payment_value", "proxy_ils_salary_value", "ratio_ils_payment_salary_proxy"),
        ("best_income_proxy", "incomeValue", "ratio_proxy_incomeValue"),
        ("best_income_proxy", "per_capita_income_rur_amt", "ratio_proxy_percapita"),
        ("best_income_proxy", "salary_median_in_gex_r1", "ratio_proxy_region_salary"),
        ("turn_cur_db_avg_v2", "best_income_proxy", "ratio_dbavg_proxy"),
        ("turn_cur_cr_avg_v2", "best_income_proxy", "ratio_cravg_proxy"),
        ("turn_cur_db_sum_v2", "best_income_proxy", "ratio_dbsum_proxy"),
        ("turn_cur_cr_sum_v2", "best_income_proxy", "ratio_crsum_proxy"),
        ("curr_rur_amt_cm_avg", "best_income_proxy", "ratio_balance_proxy"),
        ("total_rur_amt_cm_avg", "best_income_proxy", "ratio_total_balance_proxy"),
        ("hdb_bki_total_max_limit", "best_income_proxy", "ratio_bki_limit_proxy"),
        ("profit_income_out_rur_amt_12m", "best_income_proxy", "ratio_profit12_proxy"),
    ]
    for a, b, new_col in compare_pairs:
        if a in full.columns and b in full.columns:
            full[new_col] = safe_div(full[a], full[b])

    # Грубый high-income score как feature.
    high_parts = []
    for c in [
        "best_income_proxy", "proxy_salary_value", "proxy_payout_value", "income_like_max", "payout_like_max",
        "money_like_max", "hdb_bki_total_max_limit", "curr_rur_amt_cm_avg", "total_rur_amt_cm_avg",
    ]:
        if c in full.columns:
            s = pd.to_numeric(full[c], errors="coerce")
            high_parts.append(np.log1p(s.clip(lower=0)).to_numpy(dtype="float64"))
    if high_parts:
        full["heuristic_high_income_score"] = np.nanmean(np.vstack(high_parts), axis=0).astype("float32")

    return full


def build_features(train_df, test_df):
    train_features = train_df.drop(columns=["target", "w"]).copy()
    test_features = test_df.copy()
    n_train = len(train_features)
    full = pd.concat([train_features, test_features], ignore_index=True)

    # Даты
    for col in DATE_COLS:
        if col in full.columns:
            date = pd.to_datetime(full[col], errors="coerce")
            full[col + "_year"] = date.dt.year.astype("float32")
            full[col + "_month"] = date.dt.month.astype("float32")
            full[col + "_ordinal"] = (date.astype("int64") // 10**9 // 86400).astype("float32")
            full.loc[date.isna(), col + "_ordinal"] = np.nan

    if "dt" in full.columns and "period_last_act_ad" in full.columns:
        full["days_from_period_last_act_ad"] = (
            pd.to_datetime(full["dt"], errors="coerce")
            - pd.to_datetime(full["period_last_act_ad"], errors="coerce")
        ).dt.days.astype("float32")

    # Object: числоподобные -> float, остальные -> category + frequency.
    cat_cols = []
    for col in list(full.select_dtypes(include="object").columns):
        if col in DATE_COLS:
            full = full.drop(columns=[col])
            continue
        numeric = pd.to_numeric(full[col], errors="coerce")
        nonnull = full[col].notna().sum()
        bad_numeric = (full[col].notna() & numeric.isna()).sum()
        if nonnull > 0 and bad_numeric / max(nonnull, 1) < 0.001:
            full[col] = numeric.astype("float32")
        else:
            s = full[col].fillna("__MISSING__").astype(str)
            counts = s.value_counts()
            full[col + "_freq"] = (s.map(counts).astype("float32") / len(full))
            full[col] = s.astype("category")
            cat_cols.append(col)

    num_cols = [c for c in full.columns if c != "id" and c not in cat_cols]

    # Базовые row stats.
    full["num_missing_count"] = full[num_cols].isna().sum(axis=1).astype("float32")
    full["num_zero_count"] = (full[num_cols].fillna(1) == 0).sum(axis=1).astype("float32")

    key_missing_cols = [
        "incomeValue", "salary_6to12m_avg", "dp_ils_avg_salary_1y", "dp_ils_avg_salary_2y", "dp_ils_avg_salary_3y",
        "dp_payoutincomedata_payout_avg_3_month", "dp_payoutincomedata_payout_avg_6_month", "first_salary_income",
        "dp_ils_paymentssum_avg_6m", "dp_ils_paymentssum_avg_12m", "dp_ils_paymentssum_month_avg",
    ]
    for col in key_missing_cols:
        if col in full.columns:
            full[col + "_isna"] = full[col].isna().astype("float32")

    # v3/v6 aggregates. Здесь first_salary остаётся как обычный признак, но MoE/proxy используют safe-логику.
    income_like = [c for c in num_cols if any(k in c.lower() for k in ["salary", "income"]) and c in full.columns]
    if income_like:
        full["income_like_mean"] = full[income_like].mean(axis=1).astype("float32")
        full["income_like_max"] = full[income_like].max(axis=1).astype("float32")
        full["income_like_median"] = full[income_like].median(axis=1).astype("float32")
        full["income_like_count"] = full[income_like].notna().sum(axis=1).astype("float32")

    payout_like = [c for c in num_cols if "payout" in c.lower() and c in full.columns]
    if payout_like:
        full["payout_like_mean"] = full[payout_like].mean(axis=1).astype("float32")
        full["payout_like_max"] = full[payout_like].max(axis=1).astype("float32")
        full["payout_like_median"] = full[payout_like].median(axis=1).astype("float32")
        full["payout_like_count"] = full[payout_like].notna().sum(axis=1).astype("float32")

    money_like = [c for c in num_cols if ("sum" in c.lower() or "amt" in c.lower() or "amount" in c.lower()) and c in full.columns]
    if money_like:
        full["money_like_mean"] = full[money_like].mean(axis=1).astype("float32")
        full["money_like_max"] = full[money_like].max(axis=1).astype("float32")
        full["money_like_median"] = full[money_like].median(axis=1).astype("float32")

    # Proxy/source признаки строим после базовых aggregates.
    full = add_proxy_features(full)

    pairs = [
        ("turn_cur_cr_sum_v2", "turn_cur_db_sum_v2", "ratio_cr_db_sum"),
        ("turn_cur_cr_avg_v2", "turn_cur_db_avg_v2", "ratio_cr_db_avg"),
        ("salary_6to12m_avg", "incomeValue", "ratio_salary_income"),
        ("dp_ils_avg_salary_1y", "incomeValue", "ratio_ils_salary_income"),
        ("dp_payoutincomedata_payout_avg_3_month", "incomeValue", "ratio_payout3_income"),
        ("dp_payoutincomedata_payout_avg_6_month", "incomeValue", "ratio_payout6_income"),
        ("hdb_bki_total_max_limit", "incomeValue", "ratio_bki_limit_income"),
        ("hdb_bki_total_cc_max_limit", "hdb_bki_total_max_limit", "ratio_cc_total_limit"),
        ("hdb_bki_total_pil_max_limit", "hdb_bki_total_max_limit", "ratio_pil_total_limit"),
        ("curr_rur_amt_cm_avg", "incomeValue", "ratio_balance_income"),
        ("turn_cur_db_avg_v2", "incomeValue", "ratio_db_income"),
        ("turn_cur_cr_avg_v2", "incomeValue", "ratio_cr_income"),
    ]
    for left, right, new_col in pairs:
        if left in full.columns and right in full.columns:
            full[new_col] = safe_div(full[left], full[right])

    # slog1p для длиннохвостых числовых признаков.
    current_num_cols = [c for c in full.columns if c != "id" and c not in cat_cols and np.issubdtype(full[c].dtype, np.number)]
    log_cols = []
    for col in current_num_cols:
        s = full[col]
        notna = s.dropna()
        if len(notna) == 0 or notna.nunique() <= 20:
            continue
        if notna.abs().quantile(0.95) > 10:
            log_cols.append(col)
    log_cols = log_cols[:220]
    for col in log_cols:
        vals = full[col].astype("float32").values
        full[col + "_slog1p"] = (np.sign(vals) * np.log1p(np.abs(vals))).astype("float32")

    # Компактные типы.
    for col in full.columns:
        if col == "id" or col in cat_cols:
            continue
        if np.issubdtype(full[col].dtype, np.number):
            full[col] = full[col].astype("float32")

    X = full.iloc[:n_train].reset_index(drop=True)
    X_test = full.iloc[n_train:].reset_index(drop=True)
    return X, X_test, cat_cols

In [6]:
def add_time_target_encoding(X, X_test, cat_cols, y, w, train_dt, smoothing=120.0):
    # Только прошлые даты используются для train-строк. Для test — вся train-история.
    dt = pd.to_datetime(train_dt)
    ordered_dates = np.sort(dt.unique())
    sw = float(np.mean(w)) * smoothing

    for col in cat_cols:
        tr_vals = X[col].astype(str).values
        te = np.full(len(X), np.nan, dtype="float64")
        cum_wy, cum_w = {}, {}
        tot_wy, tot_w = 0.0, 0.0

        for d in ordered_dates:
            m = (dt == d).values
            if tot_w > 0:
                prior = tot_wy / tot_w
                s = pd.Series(tr_vals[m])
                num = s.map(cum_wy).fillna(0.0).values + prior * sw
                den = s.map(cum_w).fillna(0.0).values + sw
                te[m] = num / den
            g = pd.DataFrame({"v": tr_vals[m], "wy": w[m] * y[m], "wt": w[m]}).groupby("v").sum()
            for v, row in g.iterrows():
                cum_wy[v] = cum_wy.get(v, 0.0) + float(row["wy"])
                cum_w[v] = cum_w.get(v, 0.0) + float(row["wt"])
            tot_wy += float((w[m] * y[m]).sum())
            tot_w += float(w[m].sum())

        X[col + "_te"] = te.astype("float32")

        prior = tot_wy / max(tot_w, 1e-12)
        te_map = {v: (cum_wy[v] + prior * sw) / (cum_w[v] + sw) for v in cum_wy}
        X_test[col + "_te"] = X_test[col].astype(str).map(te_map).fillna(prior).astype("float32")
    return X, X_test

In [7]:
# =====================
# Сборка признаков + domain adaptation score
# =====================

t0 = time.time()
X_df, X_test_df, cat_cols = build_features(train, test)

y = train["target"].to_numpy(dtype=np.float64)
w = train["w"].to_numpy(dtype=np.float64)
train_dt = train["dt"].astype(str)

X_df, X_test_df = add_time_target_encoding(X_df, X_test_df, cat_cols, y, w, train_dt)

feature_cols = [c for c in X_df.columns if c != "id"]
safe_feature_cols = [c for c in feature_cols if "first_salary_income" not in c and "proxy_first_salary" not in c and "has_first_salary_trainonly" not in c]

CLIP_LO = max(0.0, weighted_quantile(y, 0.001, w))
CLIP_HI = weighted_quantile(y, 0.999, w) * 1.2

print("X:", X_df[feature_cols].shape, "| X_test:", X_test_df[feature_cols].shape)
print("cat_cols:", cat_cols)
print("safe_feature_cols:", len(safe_feature_cols), "из", len(feature_cols))
print(f"clip: [{CLIP_LO:,.0f}; {CLIP_HI:,.0f}]")
print("feature build time:", round(time.time() - t0, 1), "sec")

show_cols = [c for c in ["best_income_proxy", "proxy_type_code", "source_segment", "proxy_confidence", "proxy_payout_from", "has_first_salary_trainonly"] if c in X_df.columns]
display(X_df[show_cols].describe())
print("source_segment train:")
print(X_df["source_segment"].value_counts(dropna=False).sort_index())
print("source_segment test:")
print(X_test_df["source_segment"].value_counts(dropna=False).sort_index())

X: (76786, 516) | X_test: (73214, 516)
cat_cols: ['gender', 'adminarea', 'city_smart_name', 'dp_ewb_last_employment_position', 'addrref', 'dp_address_unique_regions']
safe_feature_cols: 508 из 516
clip: [20,010; 1,747,140]
feature build time: 4.8 sec


,best_income_proxy,proxy_type_code,source_segment,proxy_confidence,proxy_payout_from,has_first_salary_trainonly
count,7.229300e+04,76786.000000,76786.000000,76786.000000,76786.000000,76786.000000
mean,8.752402e+04,2.919634,2.978147,0.566795,-0.857552,0.112481
std,1.122404e+05,2.007251,1.904059,0.257574,0.351962,0.315960
min,3.615418e+00,-1.000000,0.000000,0.000000,-1.000000,0.000000
25%,4.436700e+04,2.000000,2.000000,0.480000,-1.000000,0.000000
50%,6.501628e+04,2.000000,2.000000,0.480000,-1.000000,0.000000
75%,1.022666e+05,5.000000,5.000000,0.840000,-1.000000,0.000000
max,1.667603e+07,6.000000,6.000000,0.920000,2.000000,1.000000


source_segment train:
source_segment
0.0     4493
1.0     7621
2.0    37584
3.0      228
4.0     3968
5.0     8017
6.0    14875
Name: count, dtype: int64
source_segment test:
source_segment
0.0     4862
1.0     9308
2.0    33279
3.0      303
4.0     3882
5.0    13946
6.0     7634
Name: count, dtype: int64


In [8]:
def add_domain_score(X, X_test, base_cols):
    """train-vs-test classifier: даёт модели признак похожести строки на test."""
    print("Training domain classifier...")
    t0 = time.time()
    n_tr, n_te = len(X), len(X_test)
    Z = pd.concat([X[base_cols], X_test[base_cols]], ignore_index=True)
    y_dom = np.r_[np.zeros(n_tr, dtype=np.int8), np.ones(n_te, dtype=np.int8)]
    w_dom = np.r_[np.full(n_tr, 0.5 / n_tr), np.full(n_te, 0.5 / n_te)]

    clf = LGBMClassifier(
        objective="binary",
        n_estimators=450 if not FAST_MODE else 120,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=80,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.75,
        reg_alpha=0.5,
        reg_lambda=10.0,
        max_bin=MAX_BIN,
        n_jobs=4,
        random_state=919,
        verbosity=-1,
    )
    clf.fit(Z, y_dom, sample_weight=w_dom)
    X["domain_test_like"] = clf.predict_proba(X[base_cols])[:, 1].astype("float32")
    X_test["domain_test_like"] = clf.predict_proba(X_test[base_cols])[:, 1].astype("float32")
    print("domain score:", round(time.time() - t0, 1), "sec",
          "| train mean", round(float(X["domain_test_like"].mean()), 4),
          "| test mean", round(float(X_test["domain_test_like"].mean()), 4))
    return X, X_test

X_df, X_test_df = add_domain_score(X_df, X_test_df, feature_cols)
feature_cols = [c for c in X_df.columns if c != "id"]
safe_feature_cols = [c for c in feature_cols if "first_salary_income" not in c]
print("features after domain:", len(feature_cols))

Training domain classifier...
domain score: 8.0 sec | train mean 0.5 | test mean 0.5
features after domain: 517


In [9]:
# =====================
# LightGBM helpers
# =====================

def make_lgbm(objective="regression_l1", alpha=None, seed=42, n_estimators=None,
              num_leaves=127, learning_rate=0.03, extra=None):
    params = dict(
        objective=objective,
        n_estimators=n_estimators or N_ESTIMATORS_MAX,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        min_child_samples=40,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.80,
        reg_alpha=0.2,
        reg_lambda=8.0,
        max_bin=MAX_BIN,
        n_jobs=4,
        verbosity=-1,
        random_state=seed,
    )
    if objective == "quantile":
        params["alpha"] = alpha
    if extra:
        params.update(extra)
    return LGBMRegressor(**params)


def make_classifier(seed=42, n_estimators=None, scale_pos_weight=1.0):
    return LGBMClassifier(
        objective="binary",
        n_estimators=n_estimators or (1200 if not FAST_MODE else 250),
        learning_rate=0.035,
        num_leaves=63,
        min_child_samples=60,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.78,
        reg_alpha=0.3,
        reg_lambda=10.0,
        max_bin=MAX_BIN,
        n_jobs=4,
        verbosity=-1,
        random_state=seed,
        scale_pos_weight=float(scale_pos_weight),
    )


def powered_weights(weights, power=1.0):
    weights = np.asarray(weights, dtype=np.float64)
    if power == 1.0:
        return weights
    out = np.power(np.maximum(weights, 1e-9), power)
    out = out * (np.mean(weights) / max(np.mean(out), 1e-12))
    return out


def fit_lgbm(X_tr, y_tr, w_tr, X_va=None, y_va=None, w_va=None,
             objective="regression_l1", alpha=None, seed=42,
             n_estimators=None, log_target=False, weight_power=1.0,
             num_leaves=127, learning_rate=0.03, extra=None):
    model = make_lgbm(objective, alpha, seed, n_estimators, num_leaves, learning_rate, extra)
    y_fit = np.log1p(y_tr) if log_target else y_tr
    w_fit = powered_weights(w_tr, weight_power)

    if X_va is not None:
        y_ev = np.log1p(y_va) if log_target else y_va
        model.fit(
            X_tr, y_fit, sample_weight=w_fit,
            eval_set=[(X_va, y_ev)],
            eval_sample_weight=[np.asarray(w_va, dtype=np.float64)],
            eval_metric="l1",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
        best_iter = int(model.best_iteration_ or model.n_estimators)
    else:
        model.fit(X_tr, y_fit, sample_weight=w_fit)
        best_iter = int(model.n_estimators)
    return model, best_iter


def predict_lgbm(model, X_any, log_target=False):
    p = model.predict(X_any)
    return np.expm1(p) if log_target else p


def fit_tail_classifier(X_tr, y_bin_tr, w_tr, X_va=None, y_bin_va=None, seed=42):
    pos_w = float(w_tr[y_bin_tr == 1].sum())
    neg_w = float(w_tr[y_bin_tr == 0].sum())
    spw = np.clip(neg_w / max(pos_w, 1e-6), 0.2, 50.0)
    clf = make_classifier(seed=seed, scale_pos_weight=spw)
    if X_va is not None and y_bin_va is not None:
        clf.fit(
            X_tr, y_bin_tr, sample_weight=w_tr,
            eval_set=[(X_va, y_bin_va)], eval_sample_weight=[np.ones(len(y_bin_va))],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(max(60, EARLY_STOPPING_ROUNDS // 2), verbose=False)],
        )
    else:
        clf.fit(X_tr, y_bin_tr, sample_weight=w_tr)
    return clf

In [10]:
# =====================
# Proxy-ratio, source-MoE switch, risk-MoE helpers
# =====================

SEGMENT_NAMES = {
    0: "weak",
    1: "geo",
    2: "income_value",
    3: "ils_payment",
    4: "ils_salary",
    5: "payout",
    6: "salary",
}


def get_proxy_array(frame):
    if "best_income_proxy" not in frame.columns:
        return np.full(len(frame), np.nan)
    return frame["best_income_proxy"].to_numpy(dtype=np.float64)


def get_segment_array(frame):
    if "source_segment" not in frame.columns:
        return np.zeros(len(frame), dtype=np.int8)
    return frame["source_segment"].fillna(0).astype("int8").to_numpy()


def adjusted_train_weights(weights, frame, strength=DOMAIN_WEIGHT_STRENGTH):
    """Мягко повышает вес train-строк, похожих на test по domain classifier."""
    weights = np.asarray(weights, dtype=np.float64)
    if strength <= 0 or "domain_test_like" not in frame.columns:
        return weights
    p = pd.to_numeric(frame["domain_test_like"], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(p).any():
        return weights
    p = np.nan_to_num(p, nan=np.nanmedian(p[np.isfinite(p)]))
    rank = pd.Series(p).rank(method="average", pct=True).to_numpy(dtype=np.float64)
    mult = 0.82 + strength * rank       # примерно 0.82..1.37 при strength=0.55
    out = weights * mult
    out *= np.mean(weights) / max(np.mean(out), 1e-12)
    return out


def make_test_like_oof_weights(base_w, X_oof, source_oof, source_test, domain_strength=0.45, source_strength=0.70):
    """Веса для подбора blend/stacking: ближе к распределению test по source и domain-score."""
    out = np.asarray(base_w, dtype=np.float64).copy()
    X_oof = X_oof.reset_index(drop=True)
    source_oof = np.asarray(source_oof).astype(int)
    source_test = np.asarray(source_test).astype(int)

    # Domain-rank: строки, похожие на test, важнее при подборе meta/blend.
    if "domain_test_like" in X_oof.columns and domain_strength > 0:
        p = pd.to_numeric(X_oof["domain_test_like"], errors="coerce").to_numpy(dtype=np.float64)
        if np.isfinite(p).any():
            p = np.nan_to_num(p, nan=np.nanmedian(p[np.isfinite(p)]))
            rank = pd.Series(p).rank(method="average", pct=True).to_numpy(dtype=np.float64)
            out *= (0.90 + domain_strength * rank)

    # Source distribution shift: payout/geo/weak в test могут быть представлены иначе, чем в OOF.
    vals = sorted(set(source_oof.tolist()) | set(source_test.tolist()))
    p_oof = pd.Series(source_oof).value_counts(normalize=True)
    p_test = pd.Series(source_test).value_counts(normalize=True)
    src_mult = {}
    for s in vals:
        a = float(p_test.get(s, 1e-6))
        b = float(p_oof.get(s, 1e-6))
        # sqrt сглаживает, clip не даёт одной группе захватить весь подбор.
        src_mult[int(s)] = float(np.clip((a / b) ** source_strength, 0.60, 1.85))
    out *= np.array([src_mult.get(int(s), 1.0) for s in source_oof], dtype=np.float64)

    out *= np.mean(base_w) / max(np.mean(out), 1e-12)
    return out


def fit_proxy_table(proxy_tr, seg_tr, y_tr, w_tr):
    """Простой эксперт: proxy * weighted median(target/proxy) внутри source-segment."""
    ratio = y_tr / np.where(proxy_tr > 0, proxy_tr, np.nan)
    valid = np.isfinite(ratio) & (ratio > 0) & (ratio < 50)
    out = {"global": float(weighted_median(ratio[valid], w_tr[valid])) if valid.any() else 1.0}
    if not np.isfinite(out["global"]):
        out["global"] = 1.0
    for s in sorted(np.unique(seg_tr)):
        m = (seg_tr == s) & valid
        if m.sum() >= 120:
            val = weighted_median(ratio[m], w_tr[m])
            if np.isfinite(val):
                out[int(s)] = float(np.clip(val, 0.03, 25.0))
    return out


def predict_proxy_table(proxy, seg, table, fallback):
    pred = np.array(fallback, dtype=np.float64).copy()
    good = np.isfinite(proxy) & (proxy > 0)
    for s in np.unique(seg):
        m = good & (seg == s)
        if m.any():
            r = table.get(int(s), table.get("global", 1.0))
            pred[m] = proxy[m] * r
    return pred


def fit_predict_proxy_ratio(X_tr, y_tr, w_tr, proxy_tr, X_va, y_va, w_va, proxy_va,
                            fallback_va, seed=303, n_estimators=None, train_mask=None, pred_mask=None,
                            min_rows=1800):
    if train_mask is None:
        train_mask = np.ones(len(y_tr), dtype=bool)
    if pred_mask is None:
        pred_mask = np.ones(len(y_va), dtype=bool)
    good_tr = train_mask & np.isfinite(proxy_tr) & (proxy_tr > 0) & np.isfinite(y_tr) & (y_tr > 0)
    good_va = pred_mask & np.isfinite(proxy_va) & (proxy_va > 0)
    pred = np.array(fallback_va, dtype=np.float64).copy()
    if good_tr.sum() < min_rows or good_va.sum() == 0:
        return pred, 600
    y_res = np.log1p(y_tr[good_tr]) - np.log1p(proxy_tr[good_tr])
    y_res_va = np.log1p(y_va[good_va]) - np.log1p(proxy_va[good_va])
    model, bi = fit_lgbm(
        X_tr.loc[good_tr], y_res, w_tr[good_tr],
        X_va.loc[good_va], y_res_va, w_va[good_va],
        seed=seed, n_estimators=n_estimators or N_ESTIMATORS_MAX,
        num_leaves=63, learning_rate=0.035,
        extra={"min_child_samples": 45, "reg_lambda": 14.0, "colsample_bytree": 0.82},
    )
    bi = max(int(bi), 420 if not FAST_MODE else 120)
    pred_res = model.predict(X_va.loc[good_va])
    pred[good_va] = np.expm1(np.log1p(proxy_va[good_va]) + pred_res)
    return pred, bi


def fit_predict_proxy_ratio_segment(X_tr, y_tr, w_tr, proxy_tr, seg_tr,
                                    X_va, y_va, w_va, proxy_va, seg_va,
                                    fallback_va, seed=333):
    pred = np.array(fallback_va, dtype=np.float64).copy()
    best_iters = {}
    for s in sorted(np.unique(seg_va)):
        tr_m = seg_tr == s
        va_m = seg_va == s
        if va_m.sum() == 0:
            continue
        p_tmp, bi = fit_predict_proxy_ratio(
            X_tr, y_tr, w_tr, proxy_tr, X_va, y_va, w_va, proxy_va,
            fallback_va=pred, seed=seed + int(s) * 11,
            train_mask=tr_m, pred_mask=va_m, min_rows=900 if FAST_MODE else 1200,
        )
        pred[va_m] = p_tmp[va_m]
        best_iters[int(s)] = int(bi)
    return pred, best_iters


def train_proxy_ratio_full(X_all, y_all, w_all, proxy_all, X_te, proxy_te,
                           fallback_te, seed=303, n_estimators=1200, min_rows=1800):
    good = np.isfinite(proxy_all) & (proxy_all > 0) & np.isfinite(y_all) & (y_all > 0)
    pred = np.array(fallback_te, dtype=np.float64).copy()
    good_te = np.isfinite(proxy_te) & (proxy_te > 0)
    if good.sum() < min_rows or good_te.sum() == 0:
        return pred
    y_res = np.log1p(y_all[good]) - np.log1p(proxy_all[good])
    model, _ = fit_lgbm(
        X_all.loc[good], y_res, w_all[good],
        seed=seed, n_estimators=n_estimators,
        num_leaves=63, learning_rate=0.035,
        extra={"min_child_samples": 45, "reg_lambda": 14.0, "colsample_bytree": 0.82},
    )
    pred_res = model.predict(X_te.loc[good_te])
    pred[good_te] = np.expm1(np.log1p(proxy_te[good_te]) + pred_res)
    return pred


def train_proxy_ratio_segment_full(X_all, y_all, w_all, proxy_all, seg_all,
                                   X_te, proxy_te, seg_te, fallback_te,
                                   iter_by_segment, seed=333):
    pred = np.array(fallback_te, dtype=np.float64).copy()
    for s in sorted(np.unique(seg_te)):
        tr_m = (seg_all == s) & np.isfinite(proxy_all) & (proxy_all > 0) & (y_all > 0)
        te_m = (seg_te == s) & np.isfinite(proxy_te) & (proxy_te > 0)
        if tr_m.sum() < (900 if FAST_MODE else 1200) or te_m.sum() == 0:
            continue
        y_res = np.log1p(y_all[tr_m]) - np.log1p(proxy_all[tr_m])
        n_iter = int(iter_by_segment.get(int(s), 900 if not FAST_MODE else 250))
        model, _ = fit_lgbm(
            X_all.loc[tr_m], y_res, w_all[tr_m],
            seed=seed + int(s) * 11, n_estimators=n_iter,
            num_leaves=63, learning_rate=0.035,
            extra={"min_child_samples": 45, "reg_lambda": 14.0, "colsample_bytree": 0.82},
        )
        pred_res = model.predict(X_te.loc[te_m])
        pred[te_m] = np.expm1(np.log1p(proxy_te[te_m]) + pred_res)
    return pred


def fit_predict_moe_switch(X_tr_full, y_tr, w_tr, seg_tr, X_va_full, y_va, w_va, seg_va, fallback_va, seed=707):
    """Source-MoE switch: expert учится на своём source-сегменте и применяется только к нему."""
    switch = np.array(fallback_va, dtype=np.float64).copy()
    best_iters = {}
    X_tr_safe = X_tr_full[safe_feature_cols]
    X_va_safe = X_va_full[safe_feature_cols]

    for s, name in SEGMENT_NAMES.items():
        tr_m = seg_tr == s
        va_same = seg_va == s
        if tr_m.sum() < MOE_MIN_TRAIN_ROWS or va_same.sum() == 0:
            continue
        t0 = time.time()
        model, bi = fit_lgbm(
            X_tr_safe.loc[tr_m], y_tr[tr_m], w_tr[tr_m],
            X_va_safe.loc[va_same], y_va[va_same], w_va[va_same],
            seed=seed + int(s), n_estimators=MOE_N_ESTIMATORS_MAX,
            num_leaves=63, learning_rate=0.035,
            extra={"min_child_samples": 35, "reg_lambda": 12.0, "colsample_bytree": 0.84},
        )
        bi = max(int(bi), 350 if not FAST_MODE else 100)
        switch[va_same] = model.predict(X_va_safe.loc[va_same])
        best_iters[int(s)] = bi
        print(f"    moe_switch_{name:<12} train={tr_m.sum():<6} val={va_same.sum():<6} iter={bi:<5} {time.time()-t0:6.1f} sec")
    return switch, best_iters


def train_moe_switch_full(X_all_full, y_all, w_all, seg_all, X_te_full, seg_te, fallback_te, iter_by_segment, seed=707):
    switch = np.array(fallback_te, dtype=np.float64).copy()
    X_all_safe = X_all_full[safe_feature_cols]
    X_te_safe = X_te_full[safe_feature_cols]

    for s, name in SEGMENT_NAMES.items():
        tr_m = seg_all == s
        te_same = seg_te == s
        if tr_m.sum() < MOE_MIN_TRAIN_ROWS or te_same.sum() == 0:
            continue
        n_iter = int(iter_by_segment.get(int(s), 850 if not FAST_MODE else 220))
        model, _ = fit_lgbm(
            X_all_safe.loc[tr_m], y_all[tr_m], w_all[tr_m],
            seed=seed + int(s), n_estimators=n_iter,
            num_leaves=63, learning_rate=0.035,
            extra={"min_child_samples": 35, "reg_lambda": 12.0, "colsample_bytree": 0.84},
        )
        switch[te_same] = model.predict(X_te_safe.loc[te_same])
    return switch


def fit_predict_tail_prob(X_tr, y_bin_tr, w_tr, X_va, y_bin_va=None, seed=42):
    """Безопасная бинарная tail-proba: если класс редкий/отсутствует, отдаёт константу."""
    y_bin_tr = np.asarray(y_bin_tr).astype(int)
    pos = int(y_bin_tr.sum())
    neg = int(len(y_bin_tr) - pos)
    if pos < 30 or neg < 30:
        return np.full(len(X_va), pos / max(len(y_bin_tr), 1), dtype=np.float32), 1
    clf = fit_tail_classifier(X_tr, y_bin_tr, w_tr, X_va, y_bin_va.astype(int) if y_bin_va is not None else None, seed=seed)
    bi = int(getattr(clf, "best_iteration_", 0) or getattr(clf, "n_estimators", 1))
    return clf.predict_proba(X_va)[:, 1].astype("float32"), bi


def fit_tail_prob_full(X_all, y_bin, w_all, X_te, seed=42):
    y_bin = np.asarray(y_bin).astype(int)
    pos = int(y_bin.sum())
    neg = int(len(y_bin) - pos)
    if pos < 30 or neg < 30:
        return np.full(len(X_te), pos / max(len(y_bin), 1), dtype=np.float32)
    clf = fit_tail_classifier(X_all, y_bin, w_all, seed=seed)
    return clf.predict_proba(X_te)[:, 1].astype("float32")


def make_tail_shift(base_pred, probs):
    """Мягкая ручная поправка по tail probabilities. Meta/blend сам решит, насколько ей верить."""
    p_low40 = np.asarray(probs.get("prob_low40", 0.0), dtype=np.float64)
    p_low50 = np.asarray(probs.get("prob_low50", 0.0), dtype=np.float64)
    p_h150 = np.asarray(probs.get("prob_high150", 0.0), dtype=np.float64)
    p_h250 = np.asarray(probs.get("prob_high250", 0.0), dtype=np.float64)
    p_h400 = np.asarray(probs.get("prob_high400", 0.0), dtype=np.float64)
    p_h700 = np.asarray(probs.get("prob_high700", 0.0), dtype=np.float64)
    factor = 1.0 - 0.10 * p_low40 - 0.06 * p_low50 + 0.08 * p_h150 + 0.16 * p_h250 + 0.30 * p_h400 + 0.45 * p_h700
    return np.asarray(base_pred, dtype=np.float64) * np.clip(factor, 0.82, 1.75)


def risk_model_weights(y_vals, base_w, kind):
    y_vals = np.asarray(y_vals, dtype=np.float64)
    base_w = np.asarray(base_w, dtype=np.float64)
    mult = np.ones(len(y_vals), dtype=np.float64)
    if kind == "low":
        mult += 2.8 * (y_vals < TAIL_THRESHOLDS["low50"])
        mult += 2.2 * (y_vals < TAIL_THRESHOLDS["low40"])
    elif kind == "high":
        mult += 1.4 * (y_vals > TAIL_THRESHOLDS["high150"])
        mult += 2.4 * (y_vals > TAIL_THRESHOLDS["high250"])
        mult += 1.4 * (y_vals > TAIL_THRESHOLDS["high400"])
    elif kind == "ultra":
        mult += 1.8 * (y_vals > TAIL_THRESHOLDS["high250"])
        mult += 4.0 * (y_vals > TAIL_THRESHOLDS["high400"])
        mult += 7.0 * (y_vals > TAIL_THRESHOLDS["high700"])
    out = base_w * mult
    out *= np.mean(base_w) / max(np.mean(out), 1e-12)
    return out


def compose_risk_moe(fallback, p_low_pred, p_high_pred, p_ultra_pred, probs):
    fallback = np.asarray(fallback, dtype=np.float64)
    w_low = np.clip(0.75 * np.asarray(probs.get("prob_low50", 0.0)) + 0.45 * np.asarray(probs.get("prob_low40", 0.0)), 0.0, 0.88)
    w_ultra = np.clip(0.75 * np.asarray(probs.get("prob_high400", 0.0)) + 0.60 * np.asarray(probs.get("prob_high700", 0.0)), 0.0, 0.90)
    w_high = np.clip(0.45 * np.asarray(probs.get("prob_high150", 0.0)) + 0.65 * np.asarray(probs.get("prob_high250", 0.0)), 0.0, 0.88)
    w_high = np.clip(w_high * (1.0 - 0.45 * w_ultra), 0.0, 0.88)
    total = np.clip(w_low + w_high + w_ultra, 0.0, 0.95)
    # Если high и low одновременно, fallback оставляет стабильность.
    pred = fallback * (1.0 - total) + np.asarray(p_low_pred) * w_low + np.asarray(p_high_pred) * w_high + np.asarray(p_ultra_pred) * w_ultra
    return pred


def fit_predict_risk_moe(X_tr, y_tr, w_tr, X_va, y_va, w_va, probs_va, fallback_va,
                         seed=1300, prefix="risk", n_iter_by_kind=None, min_rows=None):
    min_rows = min_rows or RISK_MIN_TRAIN_ROWS
    preds = {}
    iters = {}
    if len(y_tr) < min_rows:
        preds[f"{prefix}_low"] = np.array(fallback_va, dtype=np.float64).copy()
        preds[f"{prefix}_high"] = np.array(fallback_va, dtype=np.float64).copy()
        preds[f"{prefix}_ultra"] = np.array(fallback_va, dtype=np.float64).copy()
        preds[f"{prefix}_moe"] = np.array(fallback_va, dtype=np.float64).copy()
        return preds, {"low": 400, "high": 400, "ultra": 400}

    Xtr = X_tr[safe_feature_cols] if set(safe_feature_cols).issubset(X_tr.columns) else X_tr
    Xva = X_va[safe_feature_cols] if set(safe_feature_cols).issubset(X_va.columns) else X_va
    for j, kind in enumerate(["low", "high", "ultra"]):
        t0 = time.time()
        ww = risk_model_weights(y_tr, w_tr, kind)
        n_iter = None if n_iter_by_kind is None else int(n_iter_by_kind.get(kind, RISK_N_ESTIMATORS_MAX))
        model, bi = fit_lgbm(
            Xtr, y_tr, ww, Xva, y_va, w_va,
            seed=seed + 17 * j, n_estimators=n_iter or RISK_N_ESTIMATORS_MAX,
            num_leaves=63, learning_rate=0.032,
            extra={"min_child_samples": 45, "reg_lambda": 14.0, "colsample_bytree": 0.82},
        )
        bi = max(int(bi), 350 if not FAST_MODE else 110)
        preds[f"{prefix}_{kind}"] = model.predict(Xva)
        iters[kind] = bi
        print(f"    {prefix}_{kind:<8} iter={bi:<5} {time.time()-t0:6.1f} sec")
    preds[f"{prefix}_moe"] = compose_risk_moe(fallback_va, preds[f"{prefix}_low"], preds[f"{prefix}_high"], preds[f"{prefix}_ultra"], probs_va)
    return preds, iters


def train_risk_moe_full(X_all, y_all, w_all, X_te, probs_te, fallback_te,
                        iter_by_kind=None, seed=1300, prefix="risk", min_rows=None):
    min_rows = min_rows or RISK_MIN_TRAIN_ROWS
    preds = {}
    if len(y_all) < min_rows:
        preds[f"{prefix}_low"] = np.array(fallback_te, dtype=np.float64).copy()
        preds[f"{prefix}_high"] = np.array(fallback_te, dtype=np.float64).copy()
        preds[f"{prefix}_ultra"] = np.array(fallback_te, dtype=np.float64).copy()
        preds[f"{prefix}_moe"] = np.array(fallback_te, dtype=np.float64).copy()
        return preds

    Xtr = X_all[safe_feature_cols] if set(safe_feature_cols).issubset(X_all.columns) else X_all
    Xte = X_te[safe_feature_cols] if set(safe_feature_cols).issubset(X_te.columns) else X_te
    for j, kind in enumerate(["low", "high", "ultra"]):
        ww = risk_model_weights(y_all, w_all, kind)
        n_iter = int((iter_by_kind or {}).get(kind, 900 if not FAST_MODE else 250))
        model, _ = fit_lgbm(
            Xtr, y_all, ww,
            seed=seed + 17 * j, n_estimators=n_iter,
            num_leaves=63, learning_rate=0.032,
            extra={"min_child_samples": 45, "reg_lambda": 14.0, "colsample_bytree": 0.82},
        )
        preds[f"{prefix}_{kind}"] = model.predict(Xte)
    preds[f"{prefix}_moe"] = compose_risk_moe(fallback_te, preds[f"{prefix}_low"], preds[f"{prefix}_high"], preds[f"{prefix}_ultra"], probs_te)
    return preds


def make_risk_group_from_probs(probs_frame):
    """0=mid, 1=low, 2=high, 3=ultra. Используется только для calibration keys."""
    def col(name):
        if isinstance(probs_frame, pd.DataFrame):
            return probs_frame[name].to_numpy(dtype=np.float64) if name in probs_frame.columns else np.zeros(len(probs_frame))
        return np.asarray(probs_frame.get(name, 0.0), dtype=np.float64)
    n = len(probs_frame) if isinstance(probs_frame, pd.DataFrame) else len(next(iter(probs_frame.values())))
    risk = np.zeros(n, dtype=np.int8)
    low_score = 0.65 * col("prob_low50") + 0.55 * col("prob_low40")
    high_score = 0.45 * col("prob_high150") + 0.70 * col("prob_high250")
    ultra_score = 0.75 * col("prob_high400") + 0.60 * col("prob_high700")
    risk[low_score > 0.42] = 1
    risk[high_score > 0.42] = 2
    risk[ultra_score > 0.30] = 3
    return risk


def make_source_risk_key(source, risk):
    return np.asarray(source).astype(int) * 10 + np.asarray(risk).astype(int)


In [11]:
# =====================
# OOF: обучаем экспертов на прошлых месяцах, валидируем на следующем месяце
# =====================

BASE_MEMBERS = [
    "lgb_l1", "lgb_log", "lgb_recent", "lgb_q", "lgb_safe_missing",
    "proxy_table", "proxy_ratio_global", "proxy_ratio_segment", "proxy_ratio_recent",
    "moe_switch", "moe_switch_recent",
    "risk_low", "risk_high", "risk_ultra", "risk_moe", "risk_moe_recent",
    "tail_shift",
]
TAIL_PROB_COLS = [
    "prob_low40", "prob_low50", "prob_high150", "prob_high250", "prob_high400", "prob_high700"
]

all_dates = sorted(pd.to_datetime(train_dt.unique()))
val_dates = all_dates[-N_STACK_VAL_DATES:]
dt_series = pd.to_datetime(train_dt)

proxy_all = get_proxy_array(X_df)
seg_all = get_segment_array(X_df)
seg_test_all = get_segment_array(X_test_df)

val_store = {m: [] for m in BASE_MEMBERS}
prob_store = {m: [] for m in TAIL_PROB_COLS}
q_store = {a: [] for a in QUANTILE_GRID}
best_iters = {m: [] for m in BASE_MEMBERS}
best_iters["lgb_q_candidates"] = {str(a): [] for a in QUANTILE_GRID}
moe_iters_by_segment = {s: [] for s in SEGMENT_NAMES}
moe_recent_iters_by_segment = {s: [] for s in SEGMENT_NAMES}
ratio_segment_iters_by_segment = {s: [] for s in SEGMENT_NAMES}
risk_iters_by_kind = {"low": [], "high": [], "ultra": []}
risk_recent_iters_by_kind = {"low": [], "high": [], "ultra": []}
val_y_parts, val_w_parts, val_factor_parts, val_idx_parts, val_date_parts = [], [], [], [], []
report_rows = []

first_salary_missing = train["first_salary_income"].isna().values if "first_salary_income" in train.columns else np.ones(len(train), dtype=bool)

for d in val_dates:
    va = (dt_series == d).values
    tr = (dt_series < d).values
    print(f"\n=== fold {d.date()} | train {tr.sum()} rows, val {va.sum()} rows ===")

    X_tr, X_va = X_df.loc[tr, feature_cols], X_df.loc[va, feature_cols]
    y_tr, y_va = y[tr], y[va]
    w_tr, w_va = w[tr], w[va]
    w_fit_tr = adjusted_train_weights(w_tr, X_tr)
    proxy_tr, proxy_va = proxy_all[tr], proxy_all[va]
    seg_tr, seg_va = seg_all[tr], seg_all[va]

    fold_pred = {}
    fold_prob = {}

    # 1) обычный L1 + domain-weighting
    t0 = time.time()
    m_l1, bi = fit_lgbm(X_tr, y_tr, w_fit_tr, X_va, y_va, w_va, seed=42)
    fold_pred["lgb_l1"] = predict_lgbm(m_l1, X_va)
    best_iters["lgb_l1"].append(bi)
    print(f"  lgb_l1             best_iter={bi:<5} {time.time()-t0:7.1f} sec")

    # 2) log-target модель
    t0 = time.time()
    m_log, bi = fit_lgbm(X_tr, y_tr, w_fit_tr, X_va, y_va, w_va, seed=142, log_target=True,
                         objective="regression_l1", num_leaves=95, learning_rate=0.03)
    fold_pred["lgb_log"] = predict_lgbm(m_log, X_va, log_target=True)
    best_iters["lgb_log"].append(bi)
    print(f"  lgb_log             best_iter={bi:<5} {time.time()-t0:7.1f} sec")

    # 3) recent global model
    recent_dates = sorted(dt_series[tr].unique())[-N_RECENT_DATES:]
    rc = tr & dt_series.isin(recent_dates).values
    w_fit_rc = adjusted_train_weights(w[rc], X_df.loc[rc, feature_cols])
    t0 = time.time()
    m_recent, bi = fit_lgbm(X_df.loc[rc, feature_cols], y[rc], w_fit_rc, X_va, y_va, w_va,
                            seed=44, num_leaves=127, learning_rate=0.03)
    fold_pred["lgb_recent"] = predict_lgbm(m_recent, X_va)
    best_iters["lgb_recent"].append(bi)
    print(f"  lgb_recent          best_iter={bi:<5} {time.time()-t0:7.1f} sec")

    # 4) quantile grid
    for a in QUANTILE_GRID:
        t0 = time.time()
        mq, bi = fit_lgbm(X_tr, y_tr, w_fit_tr, X_va, y_va, w_va,
                          objective="quantile", alpha=a, seed=45,
                          num_leaves=127, learning_rate=0.03)
        q_store[a].append(predict_lgbm(mq, X_va))
        best_iters["lgb_q_candidates"][str(a)].append(bi)
        print(f"  lgb_q({a})          best_iter={bi:<5} {time.time()-t0:7.1f} sec")

    # 5) safe_missing модель: train только first_salary_missing + без first_salary features
    safe_tr = tr & first_salary_missing
    if safe_tr.sum() < 5000:
        safe_tr = tr
    w_fit_safe = adjusted_train_weights(w[safe_tr], X_df.loc[safe_tr, safe_feature_cols])
    t0 = time.time()
    m_safe, bi = fit_lgbm(
        X_df.loc[safe_tr, safe_feature_cols], y[safe_tr], w_fit_safe,
        X_df.loc[va, safe_feature_cols], y_va, w_va,
        seed=88, num_leaves=95, learning_rate=0.03,
    )
    fold_pred["lgb_safe_missing"] = predict_lgbm(m_safe, X_df.loc[va, safe_feature_cols])
    best_iters["lgb_safe_missing"].append(bi)
    print(f"  lgb_safe_missing    best_iter={bi:<5} {time.time()-t0:7.1f} sec | train rows={safe_tr.sum()}")

    # 6) proxy_table
    table = fit_proxy_table(proxy_tr, seg_tr, y_tr, w_fit_tr)
    fold_pred["proxy_table"] = predict_proxy_table(proxy_va, seg_va, table, fold_pred["lgb_l1"])
    best_iters["proxy_table"].append(1)

    # 7) global proxy-ratio residual model
    t0 = time.time()
    pr_pred, bi = fit_predict_proxy_ratio(
        X_tr, y_tr, w_fit_tr, proxy_tr, X_va, y_va, w_va, proxy_va,
        fallback_va=fold_pred["lgb_l1"], seed=303,
    )
    fold_pred["proxy_ratio_global"] = pr_pred
    best_iters["proxy_ratio_global"].append(bi)
    print(f"  proxy_ratio_global  iter={bi:<5} {time.time()-t0:7.1f} sec")

    # 8) segment-specific proxy-ratio residual model
    t0 = time.time()
    prs_pred, prs_bis = fit_predict_proxy_ratio_segment(
        X_tr, y_tr, w_fit_tr, proxy_tr, seg_tr,
        X_va, y_va, w_va, proxy_va, seg_va,
        fallback_va=fold_pred["lgb_l1"], seed=333,
    )
    fold_pred["proxy_ratio_segment"] = prs_pred
    best_iters["proxy_ratio_segment"].append(int(np.mean(list(prs_bis.values()))) if prs_bis else 700)
    for s, bi in prs_bis.items():
        ratio_segment_iters_by_segment[int(s)].append(int(bi))
    print(f"  proxy_ratio_segment segments={prs_bis} {time.time()-t0:7.1f} sec")

    # 9) recent proxy-ratio: учится только на ближайшей истории перед val
    recent4_dates = sorted(dt_series[tr].unique())[-N_RISK_RECENT_DATES:]
    rc4 = tr & dt_series.isin(recent4_dates).values
    w_fit_rc4 = adjusted_train_weights(w[rc4], X_df.loc[rc4, feature_cols])
    t0 = time.time()
    prr_pred, bi = fit_predict_proxy_ratio(
        X_df.loc[rc4, feature_cols], y[rc4], w_fit_rc4, proxy_all[rc4],
        X_va, y_va, w_va, proxy_va,
        fallback_va=fold_pred["lgb_l1"], seed=363, min_rows=1000 if FAST_MODE else 1700,
    )
    fold_pred["proxy_ratio_recent"] = prr_pred
    best_iters["proxy_ratio_recent"].append(bi)
    print(f"  proxy_ratio_recent  iter={bi:<5} rows={rc4.sum():<6} {time.time()-t0:7.1f} sec")

    # 10) Source MoE switch: all-history and recent-history
    t0 = time.time()
    moe_switch_pred, moe_bis = fit_predict_moe_switch(
        X_df.loc[tr], y_tr, w_fit_tr, seg_tr,
        X_df.loc[va], y_va, w_va, seg_va,
        fallback_va=fold_pred["lgb_l1"], seed=707,
    )
    fold_pred["moe_switch"] = moe_switch_pred
    best_iters["moe_switch"].append(int(np.mean(list(moe_bis.values()))) if moe_bis else 700)
    for s, bi in moe_bis.items():
        moe_iters_by_segment[int(s)].append(int(bi))
    print(f"  moe_switch          segments={moe_bis} total {time.time()-t0:7.1f} sec")

    t0 = time.time()
    moe_recent_pred, moe_recent_bis = fit_predict_moe_switch(
        X_df.loc[rc4], y[rc4], w_fit_rc4, seg_all[rc4],
        X_df.loc[va], y_va, w_va, seg_va,
        fallback_va=fold_pred["lgb_l1"], seed=757,
    )
    fold_pred["moe_switch_recent"] = moe_recent_pred
    best_iters["moe_switch_recent"].append(int(np.mean(list(moe_recent_bis.values()))) if moe_recent_bis else 700)
    for s, bi in moe_recent_bis.items():
        moe_recent_iters_by_segment[int(s)].append(int(bi))
    print(f"  moe_switch_recent   segments={moe_recent_bis} total {time.time()-t0:7.1f} sec")

    # 11) Tail binary classifiers: WMAE-risk gates
    tail_specs = [
        ("prob_low40",  y_tr < TAIL_THRESHOLDS["low40"],  y_va < TAIL_THRESHOLDS["low40"],  401),
        ("prob_low50",  y_tr < TAIL_THRESHOLDS["low50"],  y_va < TAIL_THRESHOLDS["low50"],  501),
        ("prob_high150", y_tr > TAIL_THRESHOLDS["high150"], y_va > TAIL_THRESHOLDS["high150"], 1501),
        ("prob_high250", y_tr > TAIL_THRESHOLDS["high250"], y_va > TAIL_THRESHOLDS["high250"], 2501),
        ("prob_high400", y_tr > TAIL_THRESHOLDS["high400"], y_va > TAIL_THRESHOLDS["high400"], 4001),
        ("prob_high700", y_tr > TAIL_THRESHOLDS["high700"], y_va > TAIL_THRESHOLDS["high700"], 7001),
    ]
    tail_iter_mean = []
    for name, yb_tr, yb_va, seed in tail_specs:
        t0 = time.time()
        prob, bi = fit_predict_tail_prob(X_tr, yb_tr.astype(int), w_fit_tr, X_va, yb_va.astype(int), seed=seed)
        fold_prob[name] = prob
        tail_iter_mean.append(bi)
        print(f"  {name:<16} pos_rate={yb_tr.mean():.3f} iter={bi:<5} {time.time()-t0:7.1f} sec")

    fold_pred["tail_shift"] = make_tail_shift(fold_pred["lgb_l1"], fold_prob)
    best_iters["tail_shift"].append(1)

    # 12) Risk-MoE: separate low/high/ultra experts and probability-gated mixture
    t0 = time.time()
    risk_preds, risk_bis = fit_predict_risk_moe(
        X_tr, y_tr, w_fit_tr, X_va, y_va, w_va, fold_prob,
        fallback_va=fold_pred["lgb_l1"], seed=1300, prefix="risk",
    )
    fold_pred.update(risk_preds)
    for k in ["low", "high", "ultra"]:
        risk_iters_by_kind[k].append(int(risk_bis.get(k, 600)))
    best_iters["risk_low"].append(risk_bis.get("low", 600))
    best_iters["risk_high"].append(risk_bis.get("high", 600))
    best_iters["risk_ultra"].append(risk_bis.get("ultra", 600))
    best_iters["risk_moe"].append(int(np.mean(list(risk_bis.values()))) if risk_bis else 700)
    print(f"  risk_moe            iters={risk_bis} total {time.time()-t0:7.1f} sec")

    # 13) Recent Risk-MoE
    t0 = time.time()
    risk_recent_preds, risk_recent_bis = fit_predict_risk_moe(
        X_df.loc[rc4, feature_cols], y[rc4], w_fit_rc4,
        X_va, y_va, w_va, fold_prob,
        fallback_va=fold_pred["lgb_l1"], seed=1370, prefix="risk_recent",
        min_rows=1600 if not FAST_MODE else 600,
    )
    fold_pred["risk_moe_recent"] = risk_recent_preds["risk_recent_moe"]
    for k in ["low", "high", "ultra"]:
        risk_recent_iters_by_kind[k].append(int(risk_recent_bis.get(k, 600)))
    best_iters["risk_moe_recent"].append(int(np.mean(list(risk_recent_bis.values()))) if risk_recent_bis else 700)
    print(f"  risk_moe_recent     iters={risk_recent_bis} total {time.time()-t0:7.1f} sec")

    # time trend factor как в v3/v6
    factor = time_trend_factor(
        train_dt[tr], y[tr], w[tr],
        np.full(int(va.sum()), float(pd.Timestamp(d).toordinal())),
    )

    row = {
        "val_date": str(d.date()),
        "baseline_wmed": weighted_mean_absolute_error(y_va, np.full(len(y_va), weighted_median(y_tr, w_tr)), w_va),
        "time_factor": float(factor[0]),
    }
    for m, p in fold_pred.items():
        if m in BASE_MEMBERS:
            row[m] = weighted_mean_absolute_error(y_va, np.clip(p * factor, CLIP_LO, CLIP_HI), w_va)
    for a in QUANTILE_GRID:
        row[f"q_{a}"] = weighted_mean_absolute_error(y_va, np.clip(q_store[a][-1] * factor, CLIP_LO, CLIP_HI), w_va)
    report_rows.append(row)

    for m in BASE_MEMBERS:
        if m != "lgb_q":
            val_store[m].append(fold_pred[m])
    for pcol in TAIL_PROB_COLS:
        prob_store[pcol].append(fold_prob[pcol])
    val_y_parts.append(y_va)
    val_w_parts.append(w_va)
    val_factor_parts.append(factor)
    val_idx_parts.append(np.where(va)[0])
    val_date_parts.append(np.full(va.sum(), pd.Timestamp(d)))

validation_report = pd.DataFrame(report_rows)
display(validation_report)



=== fold 2024-03-31 | train 16108 rows, val 13413 rows ===
  lgb_l1             best_iter=648      37.9 sec
  lgb_log             best_iter=784      35.7 sec
  lgb_recent          best_iter=529      32.8 sec
  lgb_q(0.5)          best_iter=1322     67.3 sec
  lgb_q(0.525)          best_iter=863      46.4 sec
  lgb_q(0.55)          best_iter=731      40.2 sec
  lgb_q(0.575)          best_iter=738      40.8 sec
  lgb_safe_missing    best_iter=446      22.1 sec | train rows=14670
  proxy_ratio_global  iter=451      16.5 sec
  proxy_ratio_segment segments={0: 600, 1: 485, 2: 420, 3: 600, 4: 600, 5: 600, 6: 420}    11.7 sec
  proxy_ratio_recent  iter=644   rows=16108     21.6 sec
    moe_switch_geo          train=2355   val=1543   iter=350      1.7 sec
    moe_switch_income_value train=9664   val=5708   iter=350      7.9 sec
    moe_switch_salary       train=1924   val=3029   iter=1063     8.4 sec
  moe_switch          segments={1: 350, 2: 350, 6: 1063} total    18.0 sec
    moe_switch_geo

,val_date,baseline_wmed,time_factor,lgb_l1,lgb_log,lgb_recent,lgb_safe_missing,proxy_table,proxy_ratio_global,proxy_ratio_segment,proxy_ratio_recent,moe_switch,moe_switch_recent,tail_shift,risk_low,risk_high,risk_ultra,risk_moe,risk_moe_recent,q_0.5,q_0.525,q_0.55,q_0.575
0,2024-03-31,130760.455503,1.00,64950.455753,64275.865361,64722.919524,68160.676023,81630.806058,64281.794793,64449.039740,64411.939739,64092.581815,63781.882862,73062.517585,68341.066453,72728.892584,74761.191758,71350.873748,71293.735818,64876.459079,65036.811159,65195.684900,65496.422499
1,2024-04-30,134454.711381,1.00,62617.080569,62345.972182,62123.506476,66132.139297,83965.141306,63065.061229,63944.286204,63132.609815,62320.160901,62247.785918,74022.130518,69242.251199,71587.689697,76410.818963,71375.398693,71032.080188,62570.325270,62513.900880,62551.100487,62675.890511
2,2024-05-31,125490.194356,0.97,59322.234681,59678.841607,59784.623374,61898.117372,78971.416590,61045.302720,61944.190828,60762.286155,59580.004919,59687.391447,71303.843502,66909.689458,68472.063757,71118.905795,68993.870387,69341.107566,59605.580009,59357.275573,59700.435063,60023.753592
3,2024-06-30,128040.841702,0.97,61348.907598,62127.232094,62321.686025,61849.254970,82051.764561,61910.806608,61867.921578,62006.772007,62120.407445,62171.293812,72646.478805,70830.263823,72480.083421,75137.390858,68609.262318,67922.831871,61657.757958,61869.572156,61278.979535,61667.639010


In [12]:
# Выбираем лучший quantile alpha и собираем OOF матрицу
q_mean = {a: float(np.mean([r[f"q_{a}"] for r in report_rows])) for a in QUANTILE_GRID}
BEST_Q_ALPHA = min(q_mean, key=q_mean.get)
print("mean WMAE by alpha:", {k: round(v, 1) for k, v in q_mean.items()}, "| selected alpha =", BEST_Q_ALPHA)
val_store["lgb_q"] = [q_store[BEST_Q_ALPHA][i] for i in range(len(val_dates))]
best_iters["lgb_q"] = best_iters["lgb_q_candidates"][str(BEST_Q_ALPHA)]

# Конкатенация OOF
val_idx_all = np.concatenate(val_idx_parts)
yv = np.concatenate(val_y_parts)
wv = np.concatenate(val_w_parts)
fv = np.concatenate(val_factor_parts)
val_dates_all = np.concatenate(val_date_parts)

# Применяем time factor ко всем regression predictions.
P_oof = pd.DataFrame(index=np.arange(len(yv)))
for m in BASE_MEMBERS:
    P_oof[m] = np.concatenate(val_store[m]) * fv
for pcol in TAIL_PROB_COLS:
    P_oof[pcol] = np.concatenate(prob_store[pcol])

for m in BASE_MEMBERS:
    P_oof[m] = np.clip(P_oof[m].to_numpy(dtype=np.float64), CLIP_LO, CLIP_HI)

# Отчёт по одиночным экспертам
single_scores = {m: weighted_mean_absolute_error(yv, P_oof[m], wv) for m in BASE_MEMBERS}
print("single OOF scores:", {k: round(v, 1) for k, v in sorted(single_scores.items(), key=lambda x: x[1])})

# final iters
FINAL_ITERS = {}
iter_members = [
    "lgb_l1", "lgb_log", "lgb_recent", "lgb_q", "lgb_safe_missing",
    "proxy_ratio_global", "proxy_ratio_segment", "proxy_ratio_recent",
    "moe_switch", "moe_switch_recent",
    "risk_low", "risk_high", "risk_ultra", "risk_moe", "risk_moe_recent",
]
for m in iter_members:
    vals = best_iters.get(m, [])
    vals = [int(v) for v in vals if np.isfinite(v)]
    FINAL_ITERS[m] = max(250, int(np.mean(vals) * FINAL_ITER_MULT)) if vals else 1000

MOE_FINAL_ITERS = {}
for s in SEGMENT_NAMES:
    vals = moe_iters_by_segment.get(s, [])
    MOE_FINAL_ITERS[s] = max(250, int(np.mean(vals) * FINAL_ITER_MULT)) if vals else FINAL_ITERS.get("moe_switch", 900)

MOE_RECENT_FINAL_ITERS = {}
for s in SEGMENT_NAMES:
    vals = moe_recent_iters_by_segment.get(s, [])
    MOE_RECENT_FINAL_ITERS[s] = max(250, int(np.mean(vals) * FINAL_ITER_MULT)) if vals else FINAL_ITERS.get("moe_switch_recent", 800)

RATIO_SEG_FINAL_ITERS = {}
for s in SEGMENT_NAMES:
    vals = ratio_segment_iters_by_segment.get(s, [])
    RATIO_SEG_FINAL_ITERS[s] = max(250, int(np.mean(vals) * FINAL_ITER_MULT)) if vals else FINAL_ITERS.get("proxy_ratio_segment", 900)

RISK_FINAL_ITERS = {}
for k in ["low", "high", "ultra"]:
    vals = risk_iters_by_kind.get(k, [])
    RISK_FINAL_ITERS[k] = max(250, int(np.mean(vals) * FINAL_ITER_MULT)) if vals else FINAL_ITERS.get(f"risk_{k}", 900)

RISK_RECENT_FINAL_ITERS = {}
for k in ["low", "high", "ultra"]:
    vals = risk_recent_iters_by_kind.get(k, [])
    RISK_RECENT_FINAL_ITERS[k] = max(250, int(np.mean(vals) * FINAL_ITER_MULT)) if vals else FINAL_ITERS.get("risk_moe_recent", 850)

print("FINAL_ITERS:", FINAL_ITERS)
print("MOE_FINAL_ITERS:", MOE_FINAL_ITERS)
print("MOE_RECENT_FINAL_ITERS:", MOE_RECENT_FINAL_ITERS)
print("RATIO_SEG_FINAL_ITERS:", RATIO_SEG_FINAL_ITERS)
print("RISK_FINAL_ITERS:", RISK_FINAL_ITERS)
print("RISK_RECENT_FINAL_ITERS:", RISK_RECENT_FINAL_ITERS)


mean WMAE by alpha: {0.5: 62177.5, 0.525: 62194.4, 0.55: 62181.5, 0.575: 62465.9} | selected alpha = 0.5
single OOF scores: {'moe_switch_recent': 61881.9, 'lgb_l1': 61914.2, 'moe_switch': 61926.1, 'lgb_log': 62001.1, 'lgb_q': 62044.3, 'lgb_recent': 62125.1, 'proxy_ratio_recent': 62482.0, 'proxy_ratio_global': 62486.7, 'proxy_ratio_segment': 62968.6, 'lgb_safe_missing': 64308.5, 'risk_low': 68842.8, 'risk_moe_recent': 69810.3, 'risk_moe': 69997.3, 'risk_high': 71244.7, 'tail_shift': 72717.8, 'risk_ultra': 74293.8, 'proxy_table': 81606.3}
FINAL_ITERS: {'lgb_l1': 1131, 'lgb_log': 1498, 'lgb_recent': 697, 'lgb_q': 1474, 'lgb_safe_missing': 1194, 'proxy_ratio_global': 3124, 'proxy_ratio_segment': 822, 'proxy_ratio_recent': 2611, 'moe_switch': 651, 'moe_switch_recent': 607, 'risk_low': 2809, 'risk_high': 779, 'risk_ultra': 758, 'risk_moe': 1448, 'risk_moe_recent': 1240}
MOE_FINAL_ITERS: {0: 392, 1: 523, 2: 607, 3: 651, 4: 557, 5: 429, 6: 1224}
MOE_RECENT_FINAL_ITERS: {0: 392, 1: 585, 2: 602,

In [13]:
# =====================
# Test-like linear blend + stacking meta model
# =====================

source_oof = X_df.loc[val_idx_all, "source_segment"].fillna(0).astype(int).to_numpy() if "source_segment" in X_df.columns else np.zeros(len(yv), dtype=int)
X_oof_rows = X_df.loc[val_idx_all].reset_index(drop=True)
wv_stack = make_test_like_oof_weights(wv, X_oof_rows, source_oof, seg_test_all)
print("wv mean / wv_stack mean:", round(float(wv.mean()), 4), round(float(wv_stack.mean()), 4))
print("source OOF counts:", dict(pd.Series(source_oof).value_counts().sort_index()))
print("source TEST counts:", dict(pd.Series(seg_test_all).value_counts().sort_index()))

BLEND_MEMBERS = BASE_MEMBERS.copy()
P_mat = P_oof[BLEND_MEMBERS].to_numpy(dtype=np.float64)


def blend_obj(wts):
    pred = P_mat @ wts
    return weighted_mean_absolute_error(yv, np.clip(pred, CLIP_LO, CLIP_HI), wv_stack)

x0 = np.full(len(BLEND_MEMBERS), 1.0 / len(BLEND_MEMBERS))
res = minimize(
    blend_obj, x0, method="SLSQP",
    bounds=[(0.0, 1.0)] * len(BLEND_MEMBERS),
    constraints=[{"type": "eq", "fun": lambda t: t.sum() - 1.0}],
    options={"maxiter": 900},
)
BLEND_WEIGHTS = res.x / res.x.sum()
base_blend_oof = np.clip(P_mat @ BLEND_WEIGHTS, CLIP_LO, CLIP_HI)
print("linear blend OOF real:", weighted_mean_absolute_error(yv, base_blend_oof, wv))
print("linear blend OOF test-like:", weighted_mean_absolute_error(yv, base_blend_oof, wv_stack))
print("blend weights:", {m: round(float(v), 4) for m, v in zip(BLEND_MEMBERS, BLEND_WEIGHTS) if v > 1e-4})

# Глобальная MAE-калибровка линейного бленда, подобранная на test-like weights.
def calib_obj(p):
    a, b = p
    return weighted_mean_absolute_error(yv, np.clip(a * base_blend_oof + b, CLIP_LO, CLIP_HI), wv_stack)

res2 = minimize(calib_obj, x0=np.array([1.0, 0.0]), method="Nelder-Mead")
A_CAL, B_CAL = float(res2.x[0]), float(res2.x[1])
base_blend_oof = np.clip(A_CAL * base_blend_oof + B_CAL, CLIP_LO, CLIP_HI)
print("linear blend + global cal OOF real:", weighted_mean_absolute_error(yv, base_blend_oof, wv), f"a={A_CAL:.5f} b={B_CAL:.1f}")
print("linear blend + global cal OOF test-like:", weighted_mean_absolute_error(yv, base_blend_oof, wv_stack))

# Meta-features для stacking.
meta_df = P_oof.copy()
meta_source_cols = [
    "best_income_proxy", "best_income_proxy_log1p", "proxy_type_code", "proxy_confidence", "proxy_source_count",
    "source_segment", "has_good_proxy", "domain_test_like", "num_missing_count", "num_zero_count",
    "income_like_count", "payout_like_count", "money_like_median", "heuristic_high_income_score",
    "proxy_salary_value", "proxy_payout_value", "proxy_payout_from", "proxy_ils_salary_value", "proxy_ils_payment_value",
    "proxy_income_value", "proxy_geo_income_value", "has_first_salary_trainonly", "proxy_first_salary_value",
    "ratio_salary_payout_proxy", "ratio_payout_income_proxy", "ratio_ils_payment_salary_proxy",
]
for c in meta_source_cols:
    if c in X_df.columns:
        meta_df[c] = X_df.loc[val_idx_all, c].reset_index(drop=True).astype("float32")

# Дополнительные отношения между прогнозами.
ratio_pairs = [
    ("proxy_ratio_global", "lgb_l1", "pred_ratio_proxy_global_l1"),
    ("proxy_ratio_segment", "lgb_l1", "pred_ratio_proxy_segment_l1"),
    ("proxy_ratio_recent", "lgb_l1", "pred_ratio_proxy_recent_l1"),
    ("moe_switch", "lgb_l1", "pred_ratio_moe_l1"),
    ("moe_switch_recent", "lgb_l1", "pred_ratio_moe_recent_l1"),
    ("risk_moe", "lgb_l1", "pred_ratio_risk_moe_l1"),
    ("risk_moe_recent", "lgb_l1", "pred_ratio_risk_moe_recent_l1"),
    ("risk_high", "lgb_l1", "pred_ratio_risk_high_l1"),
    ("risk_ultra", "lgb_l1", "pred_ratio_risk_ultra_l1"),
    ("lgb_q", "lgb_l1", "pred_ratio_q_l1"),
    ("tail_shift", "lgb_l1", "pred_ratio_tail_shift_l1"),
]
for a, b, name in ratio_pairs:
    meta_df[name] = safe_div(meta_df[a], meta_df[b])

# Risk group as meta feature
meta_df["risk_group"] = make_risk_group_from_probs(P_oof).astype("float32")
meta_df["source_risk_key"] = make_source_risk_key(source_oof, meta_df["risk_group"].astype(int).to_numpy()).astype("float32")

meta_df = meta_df.replace([np.inf, -np.inf], np.nan).fillna(-999.0).astype("float32")
meta_features = list(meta_df.columns)

last_meta_date = max(val_dates_all)
meta_tr = val_dates_all < last_meta_date
meta_va = val_dates_all == last_meta_date
print("meta train rows:", int(meta_tr.sum()), "meta valid rows:", int(meta_va.sum()), "last date:", pd.Timestamp(last_meta_date).date())

meta_model = make_lgbm(
    objective="regression_l1", seed=2026, n_estimators=2400 if not FAST_MODE else 450,
    num_leaves=31, learning_rate=0.023,
    extra={"min_child_samples": 70, "reg_alpha": 1.2, "reg_lambda": 24.0, "colsample_bytree": 0.90},
)
meta_model.fit(
    meta_df.loc[meta_tr, meta_features], yv[meta_tr], sample_weight=wv_stack[meta_tr],
    eval_set=[(meta_df.loc[meta_va, meta_features], yv[meta_va])],
    eval_sample_weight=[wv_stack[meta_va]],
    eval_metric="l1",
    callbacks=[lgb.early_stopping(140 if not FAST_MODE else 40, verbose=False)],
)
META_BEST_ITER = int(meta_model.best_iteration_ or meta_model.n_estimators)
meta_valid_pred = np.clip(meta_model.predict(meta_df.loc[meta_va, meta_features]), CLIP_LO, CLIP_HI)
base_valid_pred = base_blend_oof[meta_va]
print("latest-month base real:", weighted_mean_absolute_error(yv[meta_va], base_valid_pred, wv[meta_va]))
print("latest-month meta real:", weighted_mean_absolute_error(yv[meta_va], meta_valid_pred, wv[meta_va]), "best_iter", META_BEST_ITER)
print("latest-month base test-like:", weighted_mean_absolute_error(yv[meta_va], base_valid_pred, wv_stack[meta_va]))
print("latest-month meta test-like:", weighted_mean_absolute_error(yv[meta_va], meta_valid_pred, wv_stack[meta_va]))

# Выбираем безопасную долю meta по последнему месяцу, но с test-like weights.
lambda_grid = np.linspace(0.0, 0.80, 17)
lambda_rows = []
for lam in lambda_grid:
    p = np.clip((1 - lam) * base_valid_pred + lam * meta_valid_pred, CLIP_LO, CLIP_HI)
    lambda_rows.append({
        "meta_lambda": float(lam),
        "latest_month_wmae_real": weighted_mean_absolute_error(yv[meta_va], p, wv[meta_va]),
        "latest_month_wmae_testlike": weighted_mean_absolute_error(yv[meta_va], p, wv_stack[meta_va]),
    })
lambda_report = pd.DataFrame(lambda_rows)
display(lambda_report)
META_LAMBDA = float(lambda_report.sort_values("latest_month_wmae_testlike").iloc[0]["meta_lambda"])
print("selected META_LAMBDA =", META_LAMBDA)

# Финальная meta-модель обучается на всём OOF с выбранным количеством деревьев.
meta_final = make_lgbm(
    objective="regression_l1", seed=2027, n_estimators=max(250, int(META_BEST_ITER * 1.10)),
    num_leaves=31, learning_rate=0.023,
    extra={"min_child_samples": 70, "reg_alpha": 1.2, "reg_lambda": 24.0, "colsample_bytree": 0.90},
)
meta_final.fit(meta_df[meta_features], yv, sample_weight=wv_stack)
meta_oof_insample = np.clip(meta_final.predict(meta_df[meta_features]), CLIP_LO, CLIP_HI)
stack_oof = np.clip((1 - META_LAMBDA) * base_blend_oof + META_LAMBDA * meta_oof_insample, CLIP_LO, CLIP_HI)
print("stack OOF in-sample diagnostic real:", weighted_mean_absolute_error(yv, stack_oof, wv))
print("stack OOF in-sample diagnostic test-like:", weighted_mean_absolute_error(yv, stack_oof, wv_stack))


wv mean / wv_stack mean: 0.5598 0.5598
source OOF counts: {0: np.int64(3325), 1: np.int64(5266), 2: np.int64(27920), 3: np.int64(196), 4: np.int64(3470), 5: np.int64(7550), 6: np.int64(12951)}
source TEST counts: {0: np.int64(4862), 1: np.int64(9308), 2: np.int64(33279), 3: np.int64(303), 4: np.int64(3882), 5: np.int64(13946), 6: np.int64(7634)}
linear blend OOF real: 60121.59594438411
linear blend OOF test-like: 62688.93615355063
blend weights: {'lgb_log': 0.0971, 'lgb_recent': 0.0351, 'lgb_safe_missing': 0.0836, 'proxy_ratio_global': 0.1288, 'proxy_ratio_segment': 0.0184, 'proxy_ratio_recent': 0.1985, 'moe_switch': 0.081, 'moe_switch_recent': 0.184, 'risk_moe': 0.0082, 'risk_moe_recent': 0.1652}
linear blend + global cal OOF real: 60128.32664560721 a=1.01191 b=-745.2
linear blend + global cal OOF test-like: 62675.78430594946
meta train rows: 44464 meta valid rows: 16214 last date: 2024-06-30
latest-month base real: 59931.58509806615
latest-month meta real: 59190.11390986536 best_iter

,meta_lambda,latest_month_wmae_real,latest_month_wmae_testlike
0,0.00,59931.585098,61643.571112
1,0.05,59803.363345,61511.903765
2,0.10,59687.061151,61391.815923
3,0.15,59583.222532,61284.382115
4,0.20,59492.875453,61190.413797
5,0.25,59408.926074,61103.158437
6,0.30,59337.381315,61027.868312
7,0.35,59275.355424,60962.222460
8,0.40,59220.962544,60904.058723
9,0.45,59173.358176,60853.178273


selected META_LAMBDA = 0.75
stack OOF in-sample diagnostic real: 55138.525661890606
stack OOF in-sample diagnostic test-like: 57549.32674302851


In [14]:
# =====================
# Source × risk-group × prediction-bin calibration + stronger local KNN residual calibration
# =====================

def fit_2d_calibrator(pred, y_true, weights, source_key, n_bins=10, shrink_weight=420.0, corr_clip=26000.0):
    pred = np.asarray(pred, dtype=np.float64)
    resid = np.asarray(y_true, dtype=np.float64) - pred
    weights = np.asarray(weights, dtype=np.float64)
    source_key = np.asarray(source_key).astype(int)

    qs = np.linspace(0, 1, n_bins + 1)
    edges = weighted_quantile_many(pred, qs, weights)
    edges[0] = -np.inf
    edges[-1] = np.inf
    for i in range(1, len(edges)):
        if not np.isfinite(edges[i]) or edges[i] <= edges[i-1]:
            edges[i] = edges[i-1] + 1e-6

    bins = np.digitize(pred, edges[1:-1], right=True)
    table = {}
    global_corr = np.zeros(n_bins, dtype=np.float64)

    for b in range(n_bins):
        m = bins == b
        if m.sum() >= 50 and weights[m].sum() > 0:
            raw = weighted_median(resid[m], weights[m])
            shrink = weights[m].sum() / (weights[m].sum() + shrink_weight)
            global_corr[b] = float(np.clip(raw * shrink, -corr_clip, corr_clip))

    for k in sorted(np.unique(source_key)):
        for b in range(n_bins):
            m = (source_key == k) & (bins == b)
            if m.sum() >= 35 and weights[m].sum() > 0:
                raw = weighted_median(resid[m], weights[m])
                shrink = weights[m].sum() / (weights[m].sum() + shrink_weight * 1.35)
                table[(int(k), int(b))] = float(np.clip(raw * shrink, -corr_clip, corr_clip))

    return {
        "edges": [float(x) for x in edges],
        "global_corr": [float(x) for x in global_corr],
        "table": {f"{k[0]}|{k[1]}": float(v) for k, v in table.items()},
        "n_bins": int(n_bins),
    }


def apply_2d_calibrator(pred, source_key, calibrator):
    pred = np.asarray(pred, dtype=np.float64)
    source_key = np.asarray(source_key).astype(int)
    edges = np.array(calibrator["edges"], dtype=np.float64)
    global_corr = np.array(calibrator["global_corr"], dtype=np.float64)
    bins = np.digitize(pred, edges[1:-1], right=True)
    bins = np.clip(bins, 0, len(global_corr) - 1)
    corr = global_corr[bins].copy()
    table = calibrator.get("table", {})
    for i, (s, b) in enumerate(zip(source_key, bins)):
        corr[i] = table.get(f"{int(s)}|{int(b)}", corr[i])
    return corr


# strength выбираем на последнем месяце, calibrator учим на предыдущих OOF-месяцах.
risk_oof = make_risk_group_from_probs(P_oof)
cal_key_oof = make_source_risk_key(source_oof, risk_oof)
cal_train = val_dates_all < last_meta_date
cal_valid = val_dates_all == last_meta_date

meta_oof_for_strength = meta_oof_insample.copy()
# На последнем месяце используем честный meta_valid_pred из модели, обученной на предыдущих OOF-месяцах.
meta_oof_for_strength[meta_va] = meta_valid_pred
precal_oof_for_strength = np.clip((1 - META_LAMBDA) * base_blend_oof + META_LAMBDA * meta_oof_for_strength, CLIP_LO, CLIP_HI)
precal_oof = np.clip((1 - META_LAMBDA) * base_blend_oof + META_LAMBDA * meta_oof_insample, CLIP_LO, CLIP_HI)

calib_pre = fit_2d_calibrator(precal_oof_for_strength[cal_train], yv[cal_train], wv_stack[cal_train], cal_key_oof[cal_train])
corr_valid = apply_2d_calibrator(precal_oof_for_strength[cal_valid], cal_key_oof[cal_valid], calib_pre)

strength_grid = [0.0, 0.10, 0.20, 0.30, 0.45, 0.60, 0.80]
cal_rows = []
for s in strength_grid:
    p = np.clip(precal_oof_for_strength[cal_valid] + s * corr_valid, CLIP_LO, CLIP_HI)
    cal_rows.append({
        "cal_strength": s,
        "latest_month_wmae_real": weighted_mean_absolute_error(yv[cal_valid], p, wv[cal_valid]),
        "latest_month_wmae_testlike": weighted_mean_absolute_error(yv[cal_valid], p, wv_stack[cal_valid]),
    })
cal_report = pd.DataFrame(cal_rows)
display(cal_report)
CAL_STRENGTH = float(cal_report.sort_values("latest_month_wmae_testlike").iloc[0]["cal_strength"])
print("selected CAL_STRENGTH =", CAL_STRENGTH)

# Итоговый 2D calibrator для test учим на всех OOF, на test-like weights.
FINAL_CALIBRATOR = fit_2d_calibrator(precal_oof, yv, wv_stack, cal_key_oof)
corr_oof_final = apply_2d_calibrator(precal_oof, cal_key_oof, FINAL_CALIBRATOR)
calibrated_oof = np.clip(precal_oof + CAL_STRENGTH * corr_oof_final, CLIP_LO, CLIP_HI)
print("2D calibrated OOF real:", weighted_mean_absolute_error(yv, calibrated_oof, wv))
print("2D calibrated OOF test-like:", weighted_mean_absolute_error(yv, calibrated_oof, wv_stack))

# =====================
# Local KNN residual/ratio calibration
# =====================

LOCAL_X_COLS = [
    # proxy/source
    "best_income_proxy", "best_income_proxy_log1p", "proxy_type_code", "proxy_confidence", "proxy_source_count",
    "source_segment", "has_good_proxy", "domain_test_like", "heuristic_high_income_score",
    "money_like_median", "income_like_count", "payout_like_count", "num_missing_count", "num_zero_count",
    # explicit labels / income categories
    "incomeValue", "incomeValueCategory", "label_Below_50k_share_r1", "label_500k_to_1M_share_r1", "label_Above_1M_share_r1",
    # source-specific proxies
    "proxy_salary_value", "proxy_payout_value", "proxy_payout_from", "proxy_ils_salary_value", "proxy_ils_payment_value",
    "proxy_income_value", "proxy_geo_income_value",
    "ratio_salary_payout_proxy", "ratio_payout_income_proxy", "ratio_ils_payment_salary_proxy",
]

LOCAL_PRED_COLS = [
    "lgb_l1", "lgb_log", "lgb_q", "lgb_safe_missing",
    "proxy_ratio_global", "proxy_ratio_segment", "proxy_ratio_recent",
    "moe_switch", "moe_switch_recent", "risk_moe_recent",
]


def make_local_cal_features(X_rows, P_rows, pred, source, risk):
    """Фичи для локальной калибровки. Только numeric; object/categorical аккуратно приводятся."""
    Xr = X_rows.reset_index(drop=True)
    Pr = P_rows.reset_index(drop=True)
    n = len(Xr)
    pred = np.asarray(pred, dtype=np.float64)
    source = np.asarray(source).astype(int)
    risk = np.asarray(risk).astype(int)

    F = pd.DataFrame(index=np.arange(n))
    F["log_pred"] = np.log1p(np.clip(pred, 0, None)).astype("float32")
    F["pred_rank"] = pd.Series(pred).rank(pct=True).to_numpy(dtype="float32")
    F["source_segment_local"] = source.astype("float32")
    F["risk_group_local"] = risk.astype("float32")
    F["source_risk_key_local"] = make_source_risk_key(source, risk).astype("float32")

    for c in LOCAL_X_COLS:
        if c in Xr.columns:
            F[c] = pd.to_numeric(Xr[c], errors="coerce").astype("float32")

    for c in TAIL_PROB_COLS:
        if c in Pr.columns:
            F[c] = pd.to_numeric(Pr[c], errors="coerce").astype("float32")

    for c in LOCAL_PRED_COLS:
        if c in Pr.columns:
            vals = pd.to_numeric(Pr[c], errors="coerce").to_numpy(dtype=np.float64)
            F[f"logpred_{c}"] = np.log1p(np.clip(vals, 0, None)).astype("float32")
            F[f"ratio_{c}_base"] = safe_div(vals, pred)

    # У явных label share маленький масштаб, усиливаем их как координаты расстояния.
    for c in ["label_Below_50k_share_r1", "label_500k_to_1M_share_r1", "label_Above_1M_share_r1"]:
        if c in F.columns:
            F[c] = F[c].astype("float32") * 3.0
    if "incomeValueCategory" in F.columns:
        F["incomeValueCategory"] = F["incomeValueCategory"].astype("float32") * 0.50
    return F.replace([np.inf, -np.inf], np.nan)


def fit_local_standardizer(F, weights=None):
    A = F.to_numpy(dtype=np.float64)
    med = np.nanmedian(A, axis=0)
    med = np.where(np.isfinite(med), med, 0.0)
    A2 = np.where(np.isfinite(A), A, med)
    q25 = np.nanpercentile(A2, 25, axis=0)
    q75 = np.nanpercentile(A2, 75, axis=0)
    scale = q75 - q25
    scale = np.where(np.isfinite(scale) & (scale > 1e-6), scale, np.nanstd(A2, axis=0))
    scale = np.where(np.isfinite(scale) & (scale > 1e-6), scale, 1.0)
    return {"columns": list(F.columns), "med": med.astype("float64"), "scale": scale.astype("float64")}


def transform_local_features(F, scaler):
    F = F.reindex(columns=scaler["columns"])
    A = F.to_numpy(dtype=np.float64)
    A = np.where(np.isfinite(A), A, scaler["med"])
    A = (A - scaler["med"]) / scaler["scale"]
    A = np.clip(A, -8.0, 8.0)
    return A.astype("float32")



def weighted_row_quantile(values, weights, q=0.50):
    """Быстрый weighted quantile по строкам для KNN residuals."""
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    if values.ndim != 2:
        values = values.reshape(1, -1)
    if weights.ndim != 2:
        weights = weights.reshape(values.shape)
    order = np.argsort(values, axis=1)
    v = np.take_along_axis(values, order, axis=1)
    wq = np.take_along_axis(weights, order, axis=1)
    wq = np.maximum(wq, 0.0)
    total = np.maximum(wq.sum(axis=1), 1e-12)
    cutoff = np.clip(float(q), 0.05, 0.95) * total
    cs = np.cumsum(wq, axis=1)
    idx = (cs >= cutoff[:, None]).argmax(axis=1)
    return v[np.arange(values.shape[0]), idx]


def local_recency_multiplier(dates, strength=LOCAL_RECENCY_STRENGTH):
    """Для local calibration новые OOF-месяцы немного важнее старых."""
    dates = pd.to_datetime(pd.Series(dates).reset_index(drop=True))
    uniq = sorted(dates.dropna().unique())
    if len(uniq) <= 1 or strength <= 0:
        return np.ones(len(dates), dtype=np.float64)
    rank_map = {d: i / max(len(uniq) - 1, 1) for i, d in enumerate(uniq)}
    r = dates.map(rank_map).fillna(0.5).to_numpy(dtype=np.float64)
    mult = 0.82 + strength * r  # примерно 0.82..1.27
    mult /= max(np.mean(mult), 1e-12)
    return mult.astype(np.float64)


def fit_local_calibrator(F_train, pred_train, y_train, weights_train, source_train, risk_train, k=LOCAL_K):
    pred_train = np.asarray(pred_train, dtype=np.float64)
    y_train = np.asarray(y_train, dtype=np.float64)
    weights_train = np.asarray(weights_train, dtype=np.float64)
    source_train = np.asarray(source_train).astype(int)
    risk_train = np.asarray(risk_train).astype(int)

    good = np.isfinite(pred_train) & np.isfinite(y_train) & (pred_train > 0) & (weights_train > 0)
    if good.sum() < LOCAL_MIN_TRAIN_ROWS:
        print("Local calibrator disabled: too few rows", int(good.sum()))
        return {"enabled": False}

    Fg = F_train.loc[good].reset_index(drop=True)
    scaler = fit_local_standardizer(Fg, weights_train[good])
    A = transform_local_features(Fg, scaler)
    k_eff = int(min(max(40 if FAST_MODE else 90, k), len(Fg)))
    nn = NearestNeighbors(n_neighbors=k_eff, metric="euclidean", algorithm="auto")
    nn.fit(A)

    resid = np.clip(y_train[good] - pred_train[good], -LOCAL_RESID_CLIP, LOCAL_RESID_CLIP).astype("float32")
    log_resid = np.clip(
        np.log1p(y_train[good]) - np.log1p(np.clip(pred_train[good], 0, None)),
        -LOCAL_LOG_RESID_CLIP, LOCAL_LOG_RESID_CLIP,
    ).astype("float32")

    return {
        "enabled": True,
        "scaler": scaler,
        "nn": nn,
        "weights": weights_train[good].astype("float32"),
        "resid": resid,
        "log_resid": log_resid,
        "source": source_train[good].astype("int16"),
        "risk": risk_train[good].astype("int8"),
        "k": k_eff,
    }


def apply_local_calibrator(cal, F_query, pred_query, source_query, risk_query, chunk_size=LOCAL_CHUNK_SIZE, q_resid=0.50, q_log=0.50):
    pred_query = np.asarray(pred_query, dtype=np.float64)
    if not cal.get("enabled", False):
        return pred_query.copy()

    source_query = np.asarray(source_query).astype(int)
    risk_query = np.asarray(risk_query).astype(int)
    A = transform_local_features(F_query.reset_index(drop=True), cal["scaler"])
    out = np.zeros(len(A), dtype=np.float64)

    for start in range(0, len(A), chunk_size):
        end = min(start + chunk_size, len(A))
        dist, ind = cal["nn"].kneighbors(A[start:end], return_distance=True)
        # Расстояния в стандартизованном пространстве: слишком близкие не должны давать бесконечный вес.
        kw = cal["weights"][ind].astype(np.float64) / np.maximum(dist + 0.20, 0.20)

        # Поощряем совпадение source/risk, но не обнуляем остальные строки: нужна устойчивость.
        same_src = cal["source"][ind] == source_query[start:end, None]
        same_risk = cal["risk"][ind] == risk_query[start:end, None]
        kw *= np.where(same_src, 1.30, 0.82)
        kw *= np.where(same_risk, 1.18, 0.92)

        denom = np.maximum(kw.sum(axis=1), 1e-12)
        # Для WMAE условная медиана/квантиль обычно лучше среднего. Но чистая медиана
        # бывает ступенчатой, поэтому смешиваем weighted-quantile и weighted-mean.
        log_vals = cal["log_resid"][ind]
        res_vals = cal["resid"][ind]
        corr_log_mean = (kw * log_vals).sum(axis=1) / denom
        corr_resid_mean = (kw * res_vals).sum(axis=1) / denom
        corr_log_q = weighted_row_quantile(log_vals, kw, q=q_log)
        corr_resid_q = weighted_row_quantile(res_vals, kw, q=q_resid)
        corr_log = 0.68 * corr_log_q + 0.32 * corr_log_mean
        corr_resid = 0.68 * corr_resid_q + 0.32 * corr_resid_mean

        # Доверие к локальной поправке падает, если ближайшие соседи далеко.
        mean_dist = np.average(dist, weights=kw, axis=1)
        shrink = np.clip(1.15 - mean_dist / 10.0, 0.35, 1.00)
        corr_log = np.clip(corr_log * shrink, -LOCAL_LOG_RESID_CLIP, LOCAL_LOG_RESID_CLIP)
        corr_resid = np.clip(corr_resid * shrink, -LOCAL_RESID_CLIP, LOCAL_RESID_CLIP)

        p = pred_query[start:end]
        p_log = np.expm1(np.log1p(np.clip(p, 0, None)) + corr_log)
        p_res = p + corr_resid
        local = 0.72 * p_log + 0.28 * p_res
        # Не даём KNN разнести прогноз слишком далеко от базовой модели.
        lo = np.maximum(CLIP_LO, p * 0.62)
        hi = np.minimum(CLIP_HI, p * 1.70 + 20000.0)
        out[start:end] = np.clip(local, lo, hi)
    return np.clip(out, CLIP_LO, CLIP_HI)


# Честный подбор силы local calibration на последнем месяце: учим local на предыдущих OOF-месяцах.
# Сначала применяем 2D calibrator, обученный только на cal_train.
train_2d_pred = np.clip(
    precal_oof_for_strength[cal_train]
    + CAL_STRENGTH * apply_2d_calibrator(precal_oof_for_strength[cal_train], cal_key_oof[cal_train], calib_pre),
    CLIP_LO, CLIP_HI,
)
valid_2d_pred = np.clip(
    precal_oof_for_strength[cal_valid]
    + CAL_STRENGTH * apply_2d_calibrator(precal_oof_for_strength[cal_valid], cal_key_oof[cal_valid], calib_pre),
    CLIP_LO, CLIP_HI,
)

F_local_train = make_local_cal_features(
    X_oof_rows.loc[cal_train], P_oof.loc[cal_train], train_2d_pred, source_oof[cal_train], risk_oof[cal_train]
)
F_local_valid = make_local_cal_features(
    X_oof_rows.loc[cal_valid], P_oof.loc[cal_valid], valid_2d_pred, source_oof[cal_valid], risk_oof[cal_valid]
)
# Для local calibration даём свежим OOF-месяцам больший вес.
local_w_train = wv_stack[cal_train] * local_recency_multiplier(val_dates_all[cal_train])
LOCAL_CAL_PRE = fit_local_calibrator(
    F_local_train, train_2d_pred, yv[cal_train], local_w_train, source_oof[cal_train], risk_oof[cal_train]
)

# В v11 local_strength упёрся в максимум сетки. В v12 расширяем сетку и подбираем
# ещё и локальный quantile residual: это ближе к оптимуму MAE/WMAE.
local_strength_grid = [0.0, 0.08, 0.16, 0.25, 0.35, 0.48, 0.62, 0.78, 0.92, 1.00]
local_q_grid = [0.44, 0.48, 0.50, 0.52, 0.56]
local_rows = []
local_pred_cache = {}
for qloc in local_q_grid:
    local_valid_pred_q = apply_local_calibrator(
        LOCAL_CAL_PRE, F_local_valid, valid_2d_pred, source_oof[cal_valid], risk_oof[cal_valid],
        q_resid=qloc, q_log=qloc,
    )
    local_pred_cache[qloc] = local_valid_pred_q
    for s_loc in local_strength_grid:
        p = np.clip((1 - s_loc) * valid_2d_pred + s_loc * local_valid_pred_q, CLIP_LO, CLIP_HI)
        local_rows.append({
            "local_q": qloc,
            "local_strength": s_loc,
            "latest_month_wmae_real": weighted_mean_absolute_error(yv[cal_valid], p, wv[cal_valid]),
            "latest_month_wmae_testlike": weighted_mean_absolute_error(yv[cal_valid], p, wv_stack[cal_valid]),
        })
local_report = pd.DataFrame(local_rows)
display(local_report.sort_values("latest_month_wmae_testlike").head(15))
best_local_row = local_report.sort_values("latest_month_wmae_testlike").iloc[0]
LOCAL_Q = float(best_local_row["local_q"])
LOCAL_STRENGTH = float(best_local_row["local_strength"])
print("selected LOCAL_Q =", LOCAL_Q, "| LOCAL_STRENGTH =", LOCAL_STRENGTH)

# Финальный local calibrator для test учим на всех OOF после 2D calibration.
F_local_all = make_local_cal_features(X_oof_rows, P_oof, calibrated_oof, source_oof, risk_oof)
local_w_all = wv_stack * local_recency_multiplier(val_dates_all)
FINAL_LOCAL_CALIBRATOR = fit_local_calibrator(F_local_all, calibrated_oof, yv, local_w_all, source_oof, risk_oof)
local_oof_insample = apply_local_calibrator(FINAL_LOCAL_CALIBRATOR, F_local_all, calibrated_oof, source_oof, risk_oof, q_resid=LOCAL_Q, q_log=LOCAL_Q)
final_oof_diag = np.clip((1 - LOCAL_STRENGTH) * calibrated_oof + LOCAL_STRENGTH * local_oof_insample, CLIP_LO, CLIP_HI)
print("local calibrated OOF in-sample diagnostic real:", weighted_mean_absolute_error(yv, final_oof_diag, wv))
print("local calibrated OOF in-sample diagnostic test-like:", weighted_mean_absolute_error(yv, final_oof_diag, wv_stack))


,cal_strength,latest_month_wmae_real,latest_month_wmae_testlike
0,0.00,59063.096470,60722.530980
1,0.10,59059.770849,60719.239318
2,0.20,59057.703857,60717.307797
3,0.30,59055.976506,60715.740935
4,0.45,59054.437189,60714.544359
5,0.60,59053.728167,60714.205024
6,0.80,59055.826618,60716.876741


selected CAL_STRENGTH = 0.6
2D calibrated OOF real: 55047.0220668365
2D calibrated OOF test-like: 57453.50235458231


,local_q,local_strength,latest_month_wmae_real,latest_month_wmae_testlike
47,0.56,0.78,58989.321840,60640.579476
46,0.56,0.62,58990.713461,60643.796787
48,0.56,0.92,58994.518281,60644.458696
49,0.56,1.00,59000.781393,60650.031918
45,0.56,0.48,58996.122616,60650.893757
44,0.56,0.35,59006.385732,60662.721903
37,0.52,0.78,59014.234689,60670.179101
36,0.52,0.62,59014.488900,60671.434097
38,0.52,0.92,59016.971970,60672.116701
43,0.56,0.25,59016.557012,60674.124609


selected LOCAL_Q = 0.56 | LOCAL_STRENGTH = 0.78
local calibrated OOF in-sample diagnostic real: 50373.00699240677
local calibrated OOF in-sample diagnostic test-like: 52491.86155543067


In [15]:
# =====================
# Final training on all train -> test predictions
# =====================

X_all = X_df[feature_cols]
X_te = X_test_df[feature_cols]
X_safe_all = X_df[safe_feature_cols]
X_safe_te = X_test_df[safe_feature_cols]
proxy_te = get_proxy_array(X_test_df)
seg_te = get_segment_array(X_test_df)
w_model_all = adjusted_train_weights(w, X_all)

test_preds = {}
test_probs = {}

print("\nFINAL TRAINING")
print("model weight multiplier: mean original", round(float(w.mean()), 4), "mean adjusted", round(float(w_model_all.mean()), 4))

# lgb_l1
t0 = time.time()
m, _ = fit_lgbm(X_all, y, w_model_all, seed=42, n_estimators=FINAL_ITERS["lgb_l1"])
test_preds["lgb_l1"] = predict_lgbm(m, X_te)
print("lgb_l1:", round(time.time() - t0, 1), "sec")

# lgb_log
t0 = time.time()
m, _ = fit_lgbm(X_all, y, w_model_all, seed=142, n_estimators=FINAL_ITERS["lgb_log"], log_target=True,
                objective="regression_l1", num_leaves=95, learning_rate=0.03)
test_preds["lgb_log"] = predict_lgbm(m, X_te, log_target=True)
print("lgb_log:", round(time.time() - t0, 1), "sec")

# recent global
recent_dates = sorted(dt_series.unique())[-N_RECENT_DATES:]
rc = dt_series.isin(recent_dates).values
w_model_rc = adjusted_train_weights(w[rc], X_df.loc[rc, feature_cols])
print("recent dates:", [str(pd.Timestamp(d).date()) for d in recent_dates], "| rows:", int(rc.sum()))
t0 = time.time()
m, _ = fit_lgbm(X_df.loc[rc, feature_cols], y[rc], w_model_rc, seed=44,
                n_estimators=FINAL_ITERS["lgb_recent"])
test_preds["lgb_recent"] = predict_lgbm(m, X_te)
print("lgb_recent:", round(time.time() - t0, 1), "sec")

# quantile
t0 = time.time()
m, _ = fit_lgbm(X_all, y, w_model_all, objective="quantile", alpha=BEST_Q_ALPHA,
                seed=45, n_estimators=FINAL_ITERS["lgb_q"])
test_preds["lgb_q"] = predict_lgbm(m, X_te)
print("lgb_q:", round(time.time() - t0, 1), "sec | alpha", BEST_Q_ALPHA)

# safe_missing
safe_mask = first_salary_missing
if safe_mask.sum() < 5000:
    safe_mask = np.ones(len(train), dtype=bool)
w_model_safe = adjusted_train_weights(w[safe_mask], X_df.loc[safe_mask, safe_feature_cols])
t0 = time.time()
m, _ = fit_lgbm(X_df.loc[safe_mask, safe_feature_cols], y[safe_mask], w_model_safe,
                seed=88, n_estimators=FINAL_ITERS["lgb_safe_missing"],
                num_leaves=95, learning_rate=0.03)
test_preds["lgb_safe_missing"] = predict_lgbm(m, X_safe_te)
print("lgb_safe_missing:", round(time.time() - t0, 1), "sec | train rows", int(safe_mask.sum()))

# proxy table
table_full = fit_proxy_table(proxy_all, seg_all, y, w_model_all)
test_preds["proxy_table"] = predict_proxy_table(proxy_te, seg_te, table_full, test_preds["lgb_l1"])
print("proxy_table ratios:", table_full)

# global proxy ratio residual
t0 = time.time()
test_preds["proxy_ratio_global"] = train_proxy_ratio_full(
    X_all, y, w_model_all, proxy_all, X_te, proxy_te,
    fallback_te=test_preds["lgb_l1"], seed=303,
    n_estimators=FINAL_ITERS["proxy_ratio_global"],
)
print("proxy_ratio_global:", round(time.time() - t0, 1), "sec")

# segment-specific proxy ratio residual
t0 = time.time()
test_preds["proxy_ratio_segment"] = train_proxy_ratio_segment_full(
    X_all, y, w_model_all, proxy_all, seg_all,
    X_te, proxy_te, seg_te,
    fallback_te=test_preds["lgb_l1"], iter_by_segment=RATIO_SEG_FINAL_ITERS, seed=333,
)
print("proxy_ratio_segment:", round(time.time() - t0, 1), "sec")

# recent proxy ratio
recent4_dates = sorted(dt_series.unique())[-N_RISK_RECENT_DATES:]
rc4 = dt_series.isin(recent4_dates).values
w_model_rc4 = adjusted_train_weights(w[rc4], X_df.loc[rc4, feature_cols])
print("recent4 dates:", [str(pd.Timestamp(d).date()) for d in recent4_dates], "| rows:", int(rc4.sum()))
t0 = time.time()
test_preds["proxy_ratio_recent"] = train_proxy_ratio_full(
    X_df.loc[rc4, feature_cols], y[rc4], w_model_rc4, proxy_all[rc4], X_te, proxy_te,
    fallback_te=test_preds["lgb_l1"], seed=363,
    n_estimators=FINAL_ITERS["proxy_ratio_recent"], min_rows=1700 if not FAST_MODE else 1000,
)
print("proxy_ratio_recent:", round(time.time() - t0, 1), "sec")

# Source-MoE switch all-history + recent
t0 = time.time()
test_preds["moe_switch"] = train_moe_switch_full(
    X_df, y, w_model_all, seg_all, X_test_df, seg_te,
    fallback_te=test_preds["lgb_l1"], iter_by_segment=MOE_FINAL_ITERS, seed=707,
)
print("moe_switch:", round(time.time() - t0, 1), "sec")

t0 = time.time()
test_preds["moe_switch_recent"] = train_moe_switch_full(
    X_df.loc[rc4], y[rc4], w_model_rc4, seg_all[rc4], X_test_df, seg_te,
    fallback_te=test_preds["lgb_l1"], iter_by_segment=MOE_RECENT_FINAL_ITERS, seed=757,
)
print("moe_switch_recent:", round(time.time() - t0, 1), "sec")

# tail classifiers full train
for name, mask, seed in [
    ("prob_low40",  y < TAIL_THRESHOLDS["low40"],  401),
    ("prob_low50",  y < TAIL_THRESHOLDS["low50"],  501),
    ("prob_high150", y > TAIL_THRESHOLDS["high150"], 1501),
    ("prob_high250", y > TAIL_THRESHOLDS["high250"], 2501),
    ("prob_high400", y > TAIL_THRESHOLDS["high400"], 4001),
    ("prob_high700", y > TAIL_THRESHOLDS["high700"], 7001),
]:
    t0 = time.time()
    test_probs[name] = fit_tail_prob_full(X_all, mask.astype(int), w_model_all, X_te, seed=seed)
    print(name, "full pos_rate", round(float(mask.mean()), 4), "|", round(time.time() - t0, 1), "sec")

test_preds["tail_shift"] = make_tail_shift(test_preds["lgb_l1"], test_probs)

# Risk-MoE full
t0 = time.time()
risk_full = train_risk_moe_full(
    X_all, y, w_model_all, X_te, test_probs, test_preds["lgb_l1"],
    iter_by_kind=RISK_FINAL_ITERS, seed=1300, prefix="risk",
)
test_preds.update(risk_full)
print("risk_moe:", round(time.time() - t0, 1), "sec")

# Recent Risk-MoE full: в BASE используем только combined prediction
t0 = time.time()
risk_recent_full = train_risk_moe_full(
    X_df.loc[rc4, feature_cols], y[rc4], w_model_rc4, X_te, test_probs, test_preds["lgb_l1"],
    iter_by_kind=RISK_RECENT_FINAL_ITERS, seed=1370, prefix="risk_recent", min_rows=1600 if not FAST_MODE else 600,
)
test_preds["risk_moe_recent"] = risk_recent_full["risk_recent_moe"]
print("risk_moe_recent:", round(time.time() - t0, 1), "sec")



FINAL TRAINING
model weight multiplier: mean original 0.569 mean adjusted 0.569
lgb_l1: 89.0 sec
lgb_log: 94.2 sec
recent dates: ['2024-04-30', '2024-05-31', '2024-06-30'] | rows: 47265
lgb_recent: 48.0 sec
lgb_q: 112.4 sec | alpha 0.5
lgb_safe_missing: 72.2 sec | train rows 68118
proxy_table ratios: {'global': 0.9794821861536023, 1: 1.001440156797576, 2: 1.1233647201598558, 3: 5.113209504925848, 4: 1.1644754851518053, 5: 0.9009982099483959, 6: 0.9536294013831343}
proxy_ratio_global: 145.4 sec
proxy_ratio_segment: 89.1 sec
recent4 dates: ['2024-03-31', '2024-04-30', '2024-05-31', '2024-06-30'] | rows: 60678
proxy_ratio_recent: 111.7 sec
moe_switch: 74.7 sec
moe_switch_recent: 66.6 sec
prob_low40 full pos_rate 0.253 | 59.7 sec
prob_low50 full pos_rate 0.3518 | 61.0 sec
prob_high150 full pos_rate 0.1258 | 62.0 sec
prob_high250 full pos_rate 0.0522 | 59.3 sec
prob_high400 full pos_rate 0.021 | 52.6 sec
prob_high700 full pos_rate 0.0072 | 42.3 sec
risk_moe: 240.0 sec
risk_moe_recent: 188.

In [16]:
# =====================
# Test stacking + final calibration + one submission.csv
# =====================

# Time factor на будущие месяцы
last_train_ord = float(pd.to_datetime(train_dt).max().toordinal())
if "dt" in test.columns:
    test_ords = dt_ordinals(test["dt"], last_train_ord)
else:
    test_ords = np.full(len(test), last_train_ord)
factor_test = time_trend_factor(train_dt, y, w, test_ords)
print("time factor test: min/median/max =",
      round(float(factor_test.min()), 4),
      round(float(np.median(factor_test)), 4),
      round(float(factor_test.max()), 4))

P_test = pd.DataFrame(index=np.arange(len(test)))
for m in BASE_MEMBERS:
    P_test[m] = np.clip(np.asarray(test_preds[m], dtype=np.float64) * factor_test, CLIP_LO, CLIP_HI)
for pcol in TAIL_PROB_COLS:
    P_test[pcol] = np.asarray(test_probs[pcol], dtype=np.float32)

base_blend_test = np.clip(P_test[BLEND_MEMBERS].to_numpy(dtype=np.float64) @ BLEND_WEIGHTS, CLIP_LO, CLIP_HI)
base_blend_test = np.clip(A_CAL * base_blend_test + B_CAL, CLIP_LO, CLIP_HI)

meta_test = P_test.copy()
for c in meta_source_cols:
    if c in X_test_df.columns:
        meta_test[c] = X_test_df[c].astype("float32")
for a, b, name in ratio_pairs:
    meta_test[name] = safe_div(meta_test[a], meta_test[b])
risk_test = make_risk_group_from_probs(P_test)
meta_test["risk_group"] = risk_test.astype("float32")
meta_test["source_risk_key"] = make_source_risk_key(seg_te, risk_test).astype("float32")
meta_test = meta_test.reindex(columns=meta_features).replace([np.inf, -np.inf], np.nan).fillna(-999.0).astype("float32")

meta_pred_test = np.clip(meta_final.predict(meta_test[meta_features]), CLIP_LO, CLIP_HI)
precal_test = np.clip((1 - META_LAMBDA) * base_blend_test + META_LAMBDA * meta_pred_test, CLIP_LO, CLIP_HI)

cal_key_test = make_source_risk_key(seg_te, risk_test)
corr_test = apply_2d_calibrator(precal_test, cal_key_test, FINAL_CALIBRATOR)
pred_2d_test = np.clip(precal_test + CAL_STRENGTH * corr_test, CLIP_LO, CLIP_HI)

F_local_test = make_local_cal_features(X_test_df, P_test, pred_2d_test, seg_te, risk_test)
local_pred_test = apply_local_calibrator(FINAL_LOCAL_CALIBRATOR, F_local_test, pred_2d_test, seg_te, risk_test, q_resid=LOCAL_Q, q_log=LOCAL_Q)
pred_test = np.clip((1 - LOCAL_STRENGTH) * pred_2d_test + LOCAL_STRENGTH * local_pred_test, CLIP_LO, CLIP_HI)

print("base_blend_test describe")
display(pd.Series(base_blend_test).describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]))
print("final pred after local calibration describe")
display(pd.Series(pred_test).describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]))
print("risk groups test:", dict(pd.Series(risk_test).value_counts().sort_index()))

submission = pd.DataFrame({"id": test["id"].values, "predict": pred_test})
submission = sample_submission[["id"]].merge(submission, on="id", how="left")

assert list(submission.columns) == ["id", "predict"]
assert submission.shape[0] == sample_submission.shape[0]
assert submission["id"].equals(sample_submission["id"])
assert submission["predict"].isna().sum() == 0
assert submission["id"].duplicated().sum() == 0

out_path = Path(SUBMISSION_NAME)
submission.to_csv(out_path, sep=";", decimal=",", index=False)
print("Saved:", out_path.resolve())
display(submission.head())


time factor test: min/median/max = 0.97 0.97 0.97
base_blend_test describe


count    7.321400e+04
mean     9.276687e+04
std      9.901833e+04
min      2.001000e+04
1%       2.305029e+04
5%       2.812868e+04
10%      3.134870e+04
25%      3.951801e+04
50%      5.948056e+04
75%      1.063379e+05
90%      1.849888e+05
95%      2.604031e+05
99%      5.449885e+05
max      1.422300e+06
dtype: float64

final pred after local calibration describe


count    7.321400e+04
mean     9.425007e+04
std      9.743858e+04
min      2.139223e+04
1%       2.404582e+04
5%       3.008669e+04
10%      3.305794e+04
25%      4.002408e+04
50%      5.801248e+04
75%      1.160979e+05
90%      1.853421e+05
95%      2.593830e+05
99%      5.233482e+05
max      1.211940e+06
dtype: float64

risk groups test: {0: np.int64(27416), 1: np.int64(38218), 2: np.int64(5907), 3: np.int64(1673)}
Saved: C:\Users\Tatya\Downloads\Telegram Desktop\Практика 1\submission.csv


,id,predict
0,0,58916.788054
1,1,39416.328524
2,3,30485.027283
3,9,66854.019760
4,11,45365.954888


In [17]:
# Диагностика параметров. Это НЕ submission, просто чтобы понимать, что выбрал ноутбук.
params = {
    "version": "v12_stronger_local_shift",
    "base_members": BASE_MEMBERS,
    "blend_members": BLEND_MEMBERS,
    "blend_weights": [float(x) for x in BLEND_WEIGHTS],
    "best_q_alpha": float(BEST_Q_ALPHA),
    "final_iters": {k: int(v) for k, v in FINAL_ITERS.items()},
    "moe_final_iters": {str(k): int(v) for k, v in MOE_FINAL_ITERS.items()},
    "moe_recent_final_iters": {str(k): int(v) for k, v in MOE_RECENT_FINAL_ITERS.items()},
    "ratio_seg_final_iters": {str(k): int(v) for k, v in RATIO_SEG_FINAL_ITERS.items()},
    "risk_final_iters": {str(k): int(v) for k, v in RISK_FINAL_ITERS.items()},
    "risk_recent_final_iters": {str(k): int(v) for k, v in RISK_RECENT_FINAL_ITERS.items()},
    "meta_best_iter": int(META_BEST_ITER),
    "meta_lambda": float(META_LAMBDA),
    "cal_strength": float(CAL_STRENGTH),
    "local_strength": float(LOCAL_STRENGTH),
    "local_q": float(LOCAL_Q),
    "local_recency_strength": float(LOCAL_RECENCY_STRENGTH),
    "local_method": "knn_weighted_quantile_mean_blend",
    "local_k": int(FINAL_LOCAL_CALIBRATOR.get("k", 0)) if isinstance(FINAL_LOCAL_CALIBRATOR, dict) else 0,
    "local_enabled": bool(FINAL_LOCAL_CALIBRATOR.get("enabled", False)) if isinstance(FINAL_LOCAL_CALIBRATOR, dict) else False,
    "local_features_count": int(len(FINAL_LOCAL_CALIBRATOR.get("scaler", {}).get("columns", []))) if isinstance(FINAL_LOCAL_CALIBRATOR, dict) and FINAL_LOCAL_CALIBRATOR.get("enabled", False) else 0,
    "global_calibration": {"a": float(A_CAL), "b": float(B_CAL)},
    "clip_lo": float(CLIP_LO),
    "clip_hi": float(CLIP_HI),
    "domain_weight_strength": float(DOMAIN_WEIGHT_STRENGTH),
    "tail_thresholds": {k: float(v) for k, v in TAIL_THRESHOLDS.items()},
    "single_oof_scores": {k: float(v) for k, v in single_scores.items()},
    "linear_blend_oof_real": float(weighted_mean_absolute_error(yv, base_blend_oof, wv)),
    "linear_blend_oof_testlike": float(weighted_mean_absolute_error(yv, base_blend_oof, wv_stack)),
    "source_segment_train_counts": {str(k): int(v) for k, v in pd.Series(seg_all).value_counts().sort_index().items()},
    "source_segment_test_counts": {str(k): int(v) for k, v in pd.Series(seg_te).value_counts().sort_index().items()},
}
with open(PARAMS_PATH, "w") as f:
    json.dump(params, f, indent=2)
print("Saved params:", PARAMS_PATH.resolve())


Saved params: C:\Users\Tatya\Downloads\Telegram Desktop\Практика 1\fitted_params_v12_stronger_local_shift.json
